# Grundfos-PIM – Import 11: Brücke PriceClaw → Marktbegleiter (vertraulich)

Übernimmt die Scrapes, die OpenClaw an **PriceClaw** (Supabase-Projekt *solarics-produktdaten*, Tabelle `offers`) liefert,
für die Marktbegleiter-Shops ins Grundfos-PIM (Schema `marktbegleiter`). **Nur gescrapte Marktpreise** (`herkunft = scrape`) –
eigene Einkaufspreise und Einzelfunde werden nie übernommen. Jede Lieferung genau einmal; regelmäßig ausführbar
(z. B. wöchentlich) – es kommt nur Neues dazu. PriceClaw wird nur gelesen.

**Vorher einmalig:** zweites Secret **`SOLARICS_DB_URL`** anlegen – Verbindungszeile von *solarics-produktdaten*
(Supabase → Projekt wechseln → Connect → Direct → Session pooler, mit Passwort) → Notebook-Zugriff einschalten.
Dazu wie bisher `SUPABASE_DB_URL` (Grundfos-PIM). **Laufzeit → Alle ausführen.**

In [ ]:
from google.colab import userdata
!pip -q install "psycopg[binary]>=3.1" openpyxl

In [ ]:
%%writefile /content/datenaudit.py
#!/usr/bin/env python3
"""
Grundfos Datenaudit – Schritt 1 (nur lesen, nichts verändern)

Untersucht:
  1. alle DATANORM-Dateien (auch in ZIPs): Hersteller, Version, Datenstand,
     Satzarten, Kurztexte, Langtexte, EAN, Preise
  2. die Grundfos-Product-Center-Installation (GPC): Dateitypen, ZIP-Inhalte,
     Datenbanken, Regeldateien, Caches
  3. die Abdeckung gegen die Grundfos-Artikelliste (806.105 Nummern)

Schreibt kleine Berichtsdateien (CSV/JSON/Markdown/TXT) in OUTPUT_DIR.
Quelldateien werden ausschließlich gelesen.

Läuft in Google Colab (Drive gemountet) oder lokal mit Google Drive für Desktop:
  python datenaudit.py --root "/content/drive/MyDrive"
  python datenaudit.py --root "G:/Meine Ablage"
"""

import argparse
import collections
import csv
import datetime as dt
import hashlib
import io
import json
import os
import re
import sqlite3
import sys
import time
import zipfile
import zlib

# ═══════════════════════════════════════════════════════════════════
# KONFIGURATION (Pfade relativ zu --root)
# ═══════════════════════════════════════════════════════════════════
DEFAULT_CONFIG = {
    "DATANORM_DIRS": ["04 Datanorm, ENF & Herstellerdaten/Datanorm X Shopify"],
    "GPC_DIRS": ["02 Grundfos/GRUNDFOS SOFTWARE X SHOPIFY/GPC"],
    "ARTICLE_LIST": "02 Grundfos/GRUNDFOS/de_DE/checkpoints/product_list.jsonl",
    "DATANORM_CSV": "02 Grundfos/Grundfos Daten X Shopify/GRUNDFOS_DATANORM_Gesamt.csv",
    "OUTPUT_DIR": "02 Grundfos/00_Datenaudit/ergebnis",
}

# Grundfos-Produktnummer: 8 Zeichen, beginnt mit 2 Ziffern (96281376, 001A5041, 05001A04)
GF_NUM = re.compile(r"(?<![0-9A-Z])([0-9]{2}[0-9A-Z]{6})(?![0-9A-Z])")
EAN_RE = re.compile(r"^\d{8}$|^\d{12,14}$")
KNOWN_MAKERS = ["GRUNDFOS", "WILO", "BRÖTJE", "BROETJE", "VIESSMANN", "BUDERUS", "VAILLANT",
                "LOWARA", "XYLEM", "KSB", "DAB", "OVENTROP", "DANFOSS", "HONEYWELL", "RESIDEO",
                "STIEBEL", "WOLF", "WEISHAUPT", "JUNKERS", "BOSCH", "GEBERIT", "GROHE", "HANSGROHE",
                "REHAU", "UPONOR", "SYR", "JUDO", "KERMI", "PURMO", "ZEHNDER", "REFLEX", "SPIROTECH",
                "CALEFFI", "AFRISO", "WATTS", "SCHELL", "KEMPER", "VIEGA", "TECE", "DORNBRACHT"]
TEXT_EXT = {".txt", ".csv", ".xml", ".json", ".ini", ".cfg", ".config", ".log", ".htm", ".html",
            ".tsv", ".dat", ".lst", ".md"}
KEY_FILE_RE = re.compile(r"replac|accessor|rule|productrange|range|spare|service|relation|link|"
                         r"cross|successor|popular|product|catalog|katalog|sortiment|\.cache$|\.db$|"
                         r"\.sdf$|\.mdb$|\.sqlite$", re.I)
MAX_DETAIL_SIMILAR = 3          # pro (Ordner, Endung) so viele Dateien im Detail
STRINGS_SCAN_BYTES = 8 * 1024 * 1024
TEXT_HEAD_LINES = 15


# ═══════════════════════════════════════════════════════════════════
# HILFSFUNKTIONEN
# ═══════════════════════════════════════════════════════════════════
def log(msg):
    print(f"[{dt.datetime.now():%H:%M:%S}] {msg}", flush=True)


def human(n):
    for unit in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {unit}"
        n /= 1024
    return f"{n:.1f} TB"


def ean_ok(code):
    """GTIN-Prüfziffer (EAN-8/UPC-12/EAN-13/GTIN-14)."""
    if not EAN_RE.match(code) or set(code) == {"0"}:
        return False
    digits = [int(c) for c in code]
    check = digits.pop()
    total = sum(d * (3 if i % 2 == 0 else 1) for i, d in enumerate(reversed(digits)))
    return (10 - total % 10) % 10 == check


def guess_encoding(raw: bytes):
    if raw.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig"
    if raw.startswith(b"\xff\xfe") or raw.startswith(b"\xfe\xff"):
        return "utf-16"
    try:
        raw.decode("utf-8")
        return "utf-8"
    except UnicodeDecodeError:
        pass
    cp850 = sum(raw.count(bytes([b])) for b in (0x81, 0x84, 0x94, 0x8E, 0x99, 0x9A, 0xE1))
    cp1252 = sum(raw.count(bytes([b])) for b in (0xFC, 0xE4, 0xF6, 0xC4, 0xD6, 0xDC, 0xDF))
    return "cp850" if cp850 > cp1252 else "cp1252"


def magic_type(head: bytes, ext: str):
    if head.startswith(b"PK\x03\x04") or head.startswith(b"PK\x05\x06"):
        return "zip"
    if head.startswith(b"SQLite format 3\x00"):
        return "sqlite"
    if head.startswith(b"\x1f\x8b"):
        return "gzip"
    if head.startswith(b"7z\xbc\xaf\x27\x1c"):
        return "7z"
    if head.startswith(b"Rar!"):
        return "rar"
    if head.startswith(b"MZ"):
        return "pe(exe/dll)"
    if head.startswith(b"\x89PNG"):
        return "png"
    if head.startswith(b"\xff\xd8"):
        return "jpg"
    if head.startswith(b"GIF8"):
        return "gif"
    if head.startswith(b"%PDF"):
        return "pdf"
    if head[:2] in (b"\x78\x9c", b"\x78\xda", b"\x78\x01"):
        return "zlib"
    if head.startswith(b"\xff\xfe") or head.startswith(b"\xfe\xff"):
        return "text-utf16"
    if head.startswith(b"MSCF"):
        return "cab"
    if head.startswith(b"\x00\x01\x00\x00Standard Jet DB") or head[4:19] == b"Standard Jet DB":
        return "access-mdb"
    stripped = head.lstrip(b"\xef\xbb\xbf \r\n\t")
    if stripped.startswith(b"<?xml") or stripped.startswith(b"<"):
        return "xml/html"
    if stripped[:1] in (b"{", b"["):
        return "json?"
    if head:
        printable = sum(32 <= b < 127 or b in (9, 10, 13) or b >= 0xC0 for b in head)
        if printable / len(head) > 0.92:
            return "text"
    if ext == ".sdf":
        return "sqlce?"
    return "binary"


def extract_strings(data: bytes, minlen=6, limit=40):
    out = []
    for m in re.finditer(rb"[\x20-\x7e\xc0-\xff]{%d,}" % minlen, data):
        out.append(m.group().decode("latin-1")[:160])
        if len(out) >= limit:
            break
    u16 = []
    for m in re.finditer(rb"(?:[\x20-\x7e]\x00){%d,}" % minlen, data):
        u16.append(m.group().decode("utf-16le")[:160])
        if len(u16) >= limit // 2:
            break
    return out, u16


def gf_numbers_in(text, bag=None):
    found = GF_NUM.findall(text.upper())
    if bag is not None:
        bag.update(found)
    return found


# ═══════════════════════════════════════════════════════════════════
# 1) DATANORM
# ═══════════════════════════════════════════════════════════════════
def looks_like_datanorm(name, head: bytes):
    n = name.lower()
    ext = os.path.splitext(n)[1]
    if "datanorm" in n or re.match(r"^\.\d{3}$", ext) or ext in (".wrg", ".rab"):
        return True
    first = head.split(b"\n", 1)[0].strip()
    return bool(re.match(rb"^V[ ;]", first))


class DatanormStats:
    def __init__(self, label):
        self.label = label
        self.md5 = hashlib.md5()
        self.size = 0
        self.lines = 0
        self.rec = collections.Counter()
        self.samples = collections.defaultdict(list)
        self.v_record = ""
        self.encoding = ""
        self.fmt = ""           # "semikolon" | "fest(128)" | "gemischt"
        self.sep_lines = 0
        self.fixed_lines = 0
        self.articles = set()
        self.with_ean = set()
        self.with_price = set()
        self.with_longkey = set()
        self.textkz = collections.Counter()
        self.longtext_keys = set()
        self.t_records_by_key = collections.Counter()
        self.maker_hits = collections.Counter()
        self.example_articles = {}      # artnr -> dict
        self.t_texts = {}               # key -> list of text pieces (nur für Beispiele)
        self.wg = collections.Counter()

    def feed_line(self, line: str):
        self.lines += 1
        if not line.strip():
            return
        rt = line[0].upper()
        is_sep = len(line) > 1 and line[1] == ";"
        if is_sep:
            self.sep_lines += 1
        else:
            self.fixed_lines += 1
        self.rec[rt] += 1
        if len(self.samples[rt]) < 4:
            self.samples[rt].append(line[:300])
        up = line.upper()
        if self.lines <= 3000:
            for m in KNOWN_MAKERS:
                if m in up:
                    self.maker_hits[m] += 1
        if rt == "V" and not self.v_record:
            self.v_record = line[:300]
            return
        if not is_sep:
            # Festformat (128 Zeichen): Satzart(1) + Verarbeitungskz.(1) + Artikelnummer(15) …
            # Vollständige Feldzerlegung erst im Import; hier nur Nummern/EAN-Indizien
            art = up[2:17].strip().replace(" ", "")
            if rt == "A" and art:
                self.articles.add(art)
            if rt == "B" and art:
                for tok in re.findall(r"\d{13}", line):
                    if ean_ok(tok):
                        self.with_ean.add(art)
                        break
            return
        f = [x.strip() for x in line.rstrip("\r\n").split(";")]
        if rt == "A" and len(f) > 3:
            art = f[2].upper().replace(" ", "")
            self.articles.add(art)
            if len(f) > 3:
                self.textkz[f[3]] += 1
            if len(f) > 9 and re.match(r"^\d+([.,]\d+)?$", f[9] or "") and f[9].strip("0"):
                self.with_price.add(art)
            if len(f) > 11 and f[11]:
                self.wg[f[11]] += 1
            if len(f) > 12 and f[12]:
                self.with_longkey.add(art)
                self.longtext_keys.add(f[12])
            if len(self.example_articles) < 400:
                self.example_articles.setdefault(art, {
                    "artikelnummer": art,
                    "kurztext1": f[4] if len(f) > 4 else "",
                    "kurztext2": f[5] if len(f) > 5 else "",
                    "preis_roh": f[9] if len(f) > 9 else "",
                    "mengeneinheit": f[8] if len(f) > 8 else "",
                    "rabattgruppe": f[10] if len(f) > 10 else "",
                    "warengruppe": f[11] if len(f) > 11 else "",
                    "langtext_schluessel": f[12] if len(f) > 12 else "",
                    "textkennzeichen": f[3],
                    "ean": "",
                })
        elif rt == "B" and len(f) > 2:
            art = f[2].upper().replace(" ", "")
            for tok in f[3:]:
                if ean_ok(tok):
                    self.with_ean.add(art)
                    if art in self.example_articles:
                        self.example_articles[art]["ean"] = tok
                    break
        elif rt == "T" and len(f) > 3:
            key = f[2]
            self.t_records_by_key[key] += 1
            if len(self.t_texts) < 3000 or key in self.t_texts:
                txt = " ".join(x for x in f[3:] if x and not x.isdigit())
                self.t_texts.setdefault(key, []).append(txt)
        elif rt == "P" and len(f) > 3:
            # P-Sätze können mehrere Artikel je Zeile enthalten
            for i in range(2, len(f) - 2):
                if f[i] and GF_NUM.fullmatch(f[i].upper()):
                    if i + 2 < len(f) and re.match(r"^\d+$", f[i + 2] or ""):
                        self.with_price.add(f[i].upper())

    def finish(self):
        if self.sep_lines and self.fixed_lines:
            self.fmt = "gemischt" if min(self.sep_lines, self.fixed_lines) > 5 else (
                "semikolon" if self.sep_lines > self.fixed_lines else "fest")
        else:
            self.fmt = "semikolon" if self.sep_lines else "fest"
        for art, ex in self.example_articles.items():
            key = ex.get("langtext_schluessel")
            if key and key in self.t_texts:
                ex["langtext_heuristisch"] = " ".join(self.t_texts[key])[:800]
            else:
                ex["langtext_heuristisch"] = ""

    def v_fields(self):
        v = self.v_record
        res = {"datum": "", "version": "", "waehrung": "", "info": ""}
        if not v:
            return res
        if len(v) > 1 and v[1] == ";":
            f = [x.strip() for x in v.split(";")]
            res["info"] = " | ".join(x for x in f[1:] if x)[:200]
            for x in f:
                if re.fullmatch(r"\d{6}|\d{8}", x) and not res["datum"]:
                    res["datum"] = x
                if re.fullmatch(r"0?[345]\d?", x) and not res["version"]:
                    res["version"] = x
                if x in ("EUR", "DEM", "CHF") and not res["waehrung"]:
                    res["waehrung"] = x
        else:
            res["datum"] = v[2:8].strip()
            res["info"] = re.sub(r"\s+", " ", v[8:123]).strip()
            res["version"] = v[123:125].strip()
            res["waehrung"] = v[125:128].strip()
        return res

    def maker(self):
        info = self.v_fields()["info"].upper()
        for m in KNOWN_MAKERS:
            if m in info:
                return m
        if self.maker_hits:
            return self.maker_hits.most_common(1)[0][0] + " (aus Texten)"
        return "unbekannt"

    def is_grundfos(self):
        return "GRUNDFOS" in self.v_record.upper() or self.maker_hits.get("GRUNDFOS", 0) >= 5


def analyse_datanorm_stream(label, fobj, size):
    st = DatanormStats(label)
    st.size = size
    head = fobj.read(256 * 1024)
    st.encoding = guess_encoding(head)
    enc = st.encoding
    if enc == "utf-16":
        # UTF-16 lässt sich nicht byteweise an \n trennen → komplett dekodieren
        data = head + fobj.read()
        st.md5.update(data)
        for ln in data.decode("utf-16", errors="replace").splitlines():
            st.feed_line(ln)
        st.finish()
        return st
    buf = head
    pending = b""
    while True:
        st.md5.update(buf)
        data = pending + buf
        lines = data.split(b"\n")
        pending = lines.pop()
        for raw in lines:
            st.feed_line(raw.decode(enc, errors="replace"))
        buf = fobj.read(4 * 1024 * 1024)
        if not buf:
            break
    if pending:
        st.feed_line(pending.decode(enc, errors="replace"))
    st.finish()
    return st


def audit_datanorm(root, cfg, out):
    log("DATANORM: suche Dateien …")
    results = []
    other_files = []
    for rel in cfg["DATANORM_DIRS"]:
        base = os.path.join(root, rel)
        if not os.path.isdir(base):
            log(f"  ⚠ Ordner nicht gefunden: {base}")
            continue
        for dp, _dn, fn in os.walk(base):
            for name in sorted(fn):
                path = os.path.join(dp, name)
                relp = os.path.relpath(path, root)
                try:
                    size = os.path.getsize(path)
                    with open(path, "rb") as fh:
                        head = fh.read(4096)
                except OSError as e:
                    other_files.append((relp, 0, f"Fehler: {e}"))
                    continue
                kind = magic_type(head, os.path.splitext(name)[1].lower())
                if kind == "zip":
                    try:
                        with zipfile.ZipFile(path) as zf:
                            for zi in zf.infolist():
                                if zi.is_dir():
                                    continue
                                with zf.open(zi) as zfh:
                                    zhead = zfh.read(4096)
                                if looks_like_datanorm(zi.filename, zhead):
                                    with zf.open(zi) as zfh:
                                        log(f"  ZIP {relp} :: {zi.filename} ({human(zi.file_size)})")
                                        results.append(analyse_datanorm_stream(
                                            f"{relp}::{zi.filename}", zfh, zi.file_size))
                                else:
                                    other_files.append((f"{relp}::{zi.filename}", zi.file_size,
                                                        magic_type(zhead, os.path.splitext(zi.filename)[1].lower())))
                    except zipfile.BadZipFile:
                        other_files.append((relp, size, "zip (defekt)"))
                elif looks_like_datanorm(name, head):
                    log(f"  {relp} ({human(size)})")
                    with open(path, "rb") as fh:
                        results.append(analyse_datanorm_stream(relp, fh, size))
                else:
                    other_files.append((relp, size, kind))

    # Duplikate per MD5
    first_by_md5 = {}
    for st in results:
        h = st.md5.hexdigest()
        st.dup_of = first_by_md5.get(h, "")
        first_by_md5.setdefault(h, st.label)

    # 01 Dateiliste
    rec_types = sorted({rt for st in results for rt in st.rec})
    with open(os.path.join(out, "01_datanorm_dateien.csv"), "w", newline="", encoding="utf-8-sig") as fh:
        w = csv.writer(fh, delimiter=";")
        w.writerow(["datei", "groesse", "md5", "duplikat_von", "hersteller", "grundfos", "datum_V",
                    "version_V", "waehrung", "info_V", "format", "encoding", "zeilen",
                    "artikel", "mit_ean", "mit_preis", "mit_langtext_schluessel", "langtext_schluessel_unique",
                    "T_saetze_schluessel"] + [f"satz_{r}" for r in rec_types])
        for st in results:
            v = st.v_fields()
            w.writerow([st.label, st.size, st.md5.hexdigest(), st.dup_of, st.maker(),
                        "ja" if st.is_grundfos() else "", v["datum"], v["version"], v["waehrung"],
                        v["info"], st.fmt, st.encoding, st.lines, len(st.articles), len(st.with_ean),
                        len(st.with_price), len(st.with_longkey), len(st.longtext_keys),
                        len(st.t_records_by_key)] + [st.rec.get(r, 0) for r in rec_types])
        for relp, size, kind in other_files:
            w.writerow([relp, size, "", "", "", "", "", "", "", f"kein DATANORM ({kind})"])

    # 02 Beispiele je eindeutiger Datei
    with open(os.path.join(out, "02_datanorm_beispiele.txt"), "w", encoding="utf-8") as fh:
        for st in results:
            if st.dup_of:
                continue
            v = st.v_fields()
            fh.write("=" * 100 + "\n")
            fh.write(f"DATEI: {st.label}\nHersteller: {st.maker()} | Grundfos: {st.is_grundfos()} | "
                     f"Datum: {v['datum']} | Version: {v['version']} | Format: {st.fmt} | "
                     f"Encoding: {st.encoding} | Zeilen: {st.lines}\n")
            fh.write(f"V-Satz: {st.v_record}\n")
            fh.write(f"Satzarten: {dict(st.rec)}\n")
            if st.textkz:
                fh.write(f"Textkennzeichen (A-Satz Feld 4): {dict(st.textkz.most_common(10))}\n")
            if st.wg:
                fh.write(f"Top-Warengruppen: {dict(st.wg.most_common(15))}\n")
            for rt, lines in sorted(st.samples.items()):
                fh.write(f"--- Beispiele Satzart {rt} ---\n")
                for ln in lines:
                    fh.write(ln.rstrip("\r\n") + "\n")
            exs = [e for e in st.example_articles.values() if e.get("langtext_heuristisch")][:3]
            if exs:
                fh.write("--- zusammengesetzte Beispielartikel (A + B + T, heuristisch) ---\n")
                for e in exs:
                    fh.write(json.dumps(e, ensure_ascii=False) + "\n")
            fh.write("\n")

    # 06 Stichprobe Grundfos-Artikel
    with open(os.path.join(out, "06_datanorm_grundfos_stichprobe.csv"), "w", newline="",
              encoding="utf-8-sig") as fh:
        cols = ["quelle", "datum_V", "artikelnummer", "kurztext1", "kurztext2", "ean", "preis_roh",
                "mengeneinheit", "rabattgruppe", "warengruppe", "textkennzeichen",
                "langtext_schluessel", "langtext_heuristisch"]
        w = csv.DictWriter(fh, fieldnames=cols, delimiter=";", extrasaction="ignore")
        w.writeheader()
        n = 0
        for st in results:
            if st.dup_of or not st.is_grundfos():
                continue
            for e in list(st.example_articles.values())[:60]:
                w.writerow({**e, "quelle": st.label, "datum_V": st.v_fields()["datum"]})
                n += 1
                if n >= 600:
                    break
    log(f"DATANORM: {len(results)} Dateien analysiert, {len(other_files)} sonstige Dateien")
    return results


# ═══════════════════════════════════════════════════════════════════
# 2) GPC (Grundfos Product Center Desktop)
# ═══════════════════════════════════════════════════════════════════
def audit_gpc(root, cfg, out, gf_numbers_from_gpc):
    log("GPC: durchsuche Installation …")
    rows = []
    groups = collections.defaultdict(list)
    for rel in cfg["GPC_DIRS"]:
        base = os.path.join(root, rel)
        if not os.path.isdir(base):
            log(f"  ⚠ Ordner nicht gefunden: {base}")
            continue
        for dp, _dn, fn in os.walk(base):
            for name in sorted(fn):
                path = os.path.join(dp, name)
                relp = os.path.relpath(path, base)
                ext = os.path.splitext(name)[1].lower()
                try:
                    size = os.path.getsize(path)
                    with open(path, "rb") as fh:
                        head = fh.read(64)
                except OSError as e:
                    rows.append([relp, 0, ext, "", f"Fehler: {e}", ""])
                    continue
                kind = magic_type(head, ext)
                rows.append([relp, size, ext, head[:16].hex(" "), kind, ""])
                groups[(os.path.dirname(relp), ext, kind)].append((path, relp, size))

    detail = []
    detail.append("# GPC – Detailanalyse\n")
    # Übersicht nach Ordner/Endung
    detail.append("## Übersicht (Ordner, Endung, Typ)\n")
    detail.append("| Ordner | Endung | Typ | Dateien | Größe gesamt |\n|---|---|---|---|---|")
    for (d, ext, kind), items in sorted(groups.items(), key=lambda kv: -sum(i[2] for i in kv[1])):
        detail.append(f"| {d or '.'} | {ext or '-'} | {kind} | {len(items)} | {human(sum(i[2] for i in items))} |")
    detail.append("")

    skip_kinds = {"pe(exe/dll)", "png", "jpg", "gif"}
    key_files = []
    for (d, ext, kind), items in sorted(groups.items()):
        # Produktnummern aus Dateinamen aller Dateien sammeln (z. B. Bilder je Artikel)
        for _p, rp, _s in items:
            gf_numbers_in(os.path.basename(rp), gf_numbers_from_gpc["dateinamen"])
            if KEY_FILE_RE.search(os.path.basename(rp)) and kind not in skip_kinds:
                key_files.append((_p, rp, _s, kind))
        if kind in skip_kinds:
            continue
        src_key = os.path.basename(items[0][1]) if len(items) == 1 else f"{d or '.'}/*{ext}"
        for path, relp, size in items[:MAX_DETAIL_SIMILAR]:
            detail.append(f"## {relp}  ({human(size)}, Typ {kind})\n")
            try:
                detail.extend(describe_file(path, size, kind, ext, gf_numbers_from_gpc, src_key))
            except Exception as e:  # noqa: BLE001 – Audit soll weiterlaufen
                detail.append(f"Fehler bei Analyse: {e!r}")
            detail.append("")
        if len(items) > MAX_DETAIL_SIMILAR:
            detail.append(f"_… weitere {len(items) - MAX_DETAIL_SIMILAR} Dateien gleicher Art in "
                          f"`{d or '.'}` ({ext}), zusammen {human(sum(i[2] for i in items))}_\n")
            # restliche Dateien der Gruppe nur nach Grundfos-Nummern durchsuchen (ohne Detailausgabe)
            if kind not in ("zip", "sqlite") and sum(i[2] for i in items) <= 300 * 1024 * 1024:
                for path, _relp, size in items[MAX_DETAIL_SIMILAR:]:
                    try:
                        with open(path, "rb") as bf:
                            data = bf.read(min(size, STRINGS_SCAN_BYTES))
                        if data[:2] in (b"\x78\x9c", b"\x78\xda"):
                            data = zlib.decompressobj().decompress(data, STRINGS_SCAN_BYTES)
                        prefix = "text:" if kind.startswith("text") or kind in ("xml/html", "json?") else "bin:"
                        gf_numbers_in(data.decode("latin-1"), gf_numbers_from_gpc[prefix + src_key])
                    except (OSError, zlib.error):
                        pass

    # Schlüsseldateien (Austausch, Zubehör, Sortiment, Service …) ausführlicher
    with open(os.path.join(out, "07_gpc_schluesseldateien.txt"), "w", encoding="utf-8") as fh:
        for path, relp, size, kind in key_files[:80]:
            fh.write("=" * 100 + f"\n{relp}  ({human(size)}, Typ {kind})\n")
            try:
                with open(path, "rb") as bf:
                    data = bf.read(min(size, STRINGS_SCAN_BYTES))
                if kind == "zip":
                    fh.write("\n".join(describe_file(path, size, kind, "", collections.defaultdict(
                        collections.Counter))) + "\n")
                    continue
                if data[:2] in (b"\x78\x9c", b"\x78\xda"):
                    try:
                        data = zlib.decompressobj().decompress(data, STRINGS_SCAN_BYTES)
                        fh.write(f"(zlib entpackt: {human(len(data))})\n")
                    except zlib.error:
                        pass
                fh.write("HEX (erste 512 Bytes):\n")
                for off in range(0, min(512, len(data)), 32):
                    chunk = data[off:off + 32]
                    fh.write(f"{off:04x}  {chunk.hex(' '):<96}  "
                             f"{''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk)}\n")
                s8, s16 = extract_strings(data, minlen=4, limit=200)
                fh.write("STRINGS:\n" + "\n".join(s8[:200]) + "\n")
                if s16:
                    fh.write("STRINGS UTF-16:\n" + "\n".join(s16[:100]) + "\n")
            except Exception as e:  # noqa: BLE001
                fh.write(f"Fehler: {e!r}\n")

    with open(os.path.join(out, "03_gpc_dateien.csv"), "w", newline="", encoding="utf-8-sig") as fh:
        w = csv.writer(fh, delimiter=";")
        w.writerow(["datei", "groesse", "endung", "magic_hex", "typ", "hinweis"])
        w.writerows(rows)
    with open(os.path.join(out, "04_gpc_details.md"), "w", encoding="utf-8") as fh:
        fh.write("\n".join(detail))
    log(f"GPC: {len(rows)} Dateien erfasst")
    return rows


def describe_file(path, size, kind, ext, gf_bag, src_key=None):
    """Beschreibt eine Datei; Grundfos-Nummern landen in gf_bag[src_key]."""
    lines = []
    key = src_key or os.path.basename(path)
    if kind == "zip":
        with zipfile.ZipFile(path) as zf:
            infos = [i for i in zf.infolist() if not i.is_dir()]
            exts = collections.Counter(os.path.splitext(i.filename)[1].lower() for i in infos)
            tops = collections.Counter(i.filename.split("/")[0] for i in infos)
            nums = collections.Counter()
            for i in infos:
                gf_numbers_in(os.path.basename(i.filename), nums)
            gf_bag["zip:" + key].update(nums)
            lines.append(f"- Einträge: {len(infos)}, entpackt: {human(sum(i.file_size for i in infos))}")
            lines.append(f"- Endungen: {dict(exts.most_common(12))}")
            lines.append(f"- Top-Ordner: {dict(tops.most_common(12))}")
            lines.append(f"- Dateinamen mit Grundfos-Nummer: {sum(nums.values())} "
                         f"({len(nums)} verschiedene Nummern)")
            lines.append("- Beispiele: " + ", ".join(f"`{i.filename}`" for i in infos[:20]))
            # kleine Text-/XML-/DB-Einträge kurz anreißen
            shown = 0
            for i in infos:
                e = os.path.splitext(i.filename)[1].lower()
                if shown < 3 and (e in TEXT_EXT or e in (".db", ".sqlite")) and i.file_size < 20_000_000:
                    with zf.open(i) as zfh:
                        raw = zfh.read(4096)
                    enc = guess_encoding(raw)
                    lines.append(f"- Kopf von `{i.filename}`:\n```\n"
                                 + "\n".join(raw.decode(enc, "replace").splitlines()[:10]) + "\n```")
                    shown += 1
    elif kind == "sqlite":
        uri = "file:" + path.replace("\\", "/") + "?mode=ro&immutable=1"
        con = sqlite3.connect(uri, uri=True)
        try:
            tabs = [r[0] for r in con.execute("select name from sqlite_master where type='table'")]
            lines.append(f"- Tabellen: {len(tabs)}")
            for t in tabs[:60]:
                cols = [r[1] for r in con.execute(f'pragma table_info("{t}")')]
                cnt = con.execute(f'select count(*) from "{t}"').fetchone()[0]
                lines.append(f"  - **{t}** ({cnt} Zeilen): {', '.join(cols)[:400]}")
                for row in con.execute(f'select * from "{t}" limit 3'):
                    lines.append("    - " + " | ".join(str(v)[:60] for v in row)[:400])
        finally:
            con.close()
    elif kind in ("text", "xml/html", "json?", "text-utf16") or ext in TEXT_EXT:
        with open(path, "rb") as fh:
            raw = fh.read(min(size, 2 * 1024 * 1024))
        enc = guess_encoding(raw)
        text = raw.decode(enc, "replace")
        nums = collections.Counter()
        gf_numbers_in(text, nums)
        gf_bag["text:" + key].update(nums)
        lines.append(f"- Encoding: {enc}, Zeilen (erste 2 MB): {text.count(chr(10))}, "
                     f"Grundfos-Nummern darin: {sum(nums.values())} ({len(nums)} verschiedene)")
        lines.append("```\n" + "\n".join(l[:300] for l in text.splitlines()[:TEXT_HEAD_LINES]) + "\n```")
    else:
        with open(path, "rb") as fh:
            data = fh.read(min(size, STRINGS_SCAN_BYTES))
        if kind == "zlib" or data[:2] in (b"\x78\x9c", b"\x78\xda"):
            try:
                data = zlib.decompress(data) if size <= STRINGS_SCAN_BYTES else \
                    zlib.decompressobj().decompress(data, STRINGS_SCAN_BYTES)
                lines.append(f"- zlib entpackt: {human(len(data))}")
            except zlib.error as e:
                lines.append(f"- zlib-Versuch fehlgeschlagen: {e}")
        elif kind == "gzip":
            import gzip
            try:
                data = gzip.decompress(data)
                lines.append(f"- gzip entpackt: {human(len(data))}")
            except Exception as e:  # noqa: BLE001
                lines.append(f"- gzip-Versuch fehlgeschlagen: {e}")
        s8, s16 = extract_strings(data)
        nums = collections.Counter()
        gf_numbers_in(data.decode("latin-1"), nums)
        gf_bag["bin:" + key].update(nums)
        lines.append(f"- Kopf (hex): `{data[:32].hex(' ')}`")
        lines.append(f"- Grundfos-Nummern im gelesenen Bereich: {sum(nums.values())} ({len(nums)} verschiedene); "
                     f"Beispiele: {', '.join(list(nums)[:10])}")
        if s8:
            lines.append("- Strings (ASCII/Latin-1): " + " ‖ ".join(s8[:30]))
        if s16:
            lines.append("- Strings (UTF-16): " + " ‖ ".join(s16[:20]))
    return lines


# ═══════════════════════════════════════════════════════════════════
# 3) ARTIKELLISTE + ABDECKUNG
# ═══════════════════════════════════════════════════════════════════
def load_article_list(path):
    arts = {}
    keys_seen = None
    if not os.path.exists(path):
        log(f"  ⚠ Artikelliste nicht gefunden: {path}")
        return arts, None
    log(f"Artikelliste laden: {path}")
    cand = ["article_number", "articleNumber", "product_number", "productNumber", "productnumber",
            "material_number", "sku", "article", "number"]
    with open(path, encoding="utf-8", errors="replace") as fh:
        for i, line in enumerate(fh):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if keys_seen is None:
                keys_seen = list(obj.keys()) if isinstance(obj, dict) else type(obj).__name__
            num = ""
            if isinstance(obj, dict):
                for k in cand:
                    if obj.get(k):
                        num = str(obj[k])
                        break
                if not num:
                    m = GF_NUM.search(json.dumps(obj).upper())
                    num = m.group(1) if m else ""
                fam = obj.get("family") or obj.get("family_slug") or ""
            else:
                fam = ""
            if num:
                arts[num.upper().replace(" ", "")] = fam
            if i % 200000 == 0 and i:
                log(f"  … {i} Zeilen")
    log(f"  {len(arts)} Artikel, Felder: {keys_seen}")
    return arts, keys_seen


def audit_datanorm_csv(path):
    res = {}
    if not os.path.exists(path):
        return res
    with open(path, encoding="utf-8-sig", errors="replace", newline="") as fh:
        r = csv.DictReader(fh, delimiter=";")
        rows = list(r)
    res["zeilen"] = len(rows)
    res["spalten"] = r.fieldnames
    res["artikel_unique"] = len({x.get("Artikelnummer", "") for x in rows})
    res["mit_langtextschluessel"] = sum(1 for x in rows if (x.get("Langtextschluessel") or "").strip())
    res["mit_ean"] = sum(1 for x in rows if ean_ok((x.get("EAN_GTIN") or "").strip()))
    res["sortimente"] = dict(collections.Counter(x.get("Sortiment", "") for x in rows).most_common(20))
    res["datenstaende"] = dict(collections.Counter(x.get("Datenstand", "") for x in rows).most_common(20))
    res["quelldateien"] = dict(collections.Counter(x.get("Quelldatei", "") for x in rows).most_common(20))
    res["_artikel"] = {x.get("Artikelnummer", "").upper().replace(" ", "") for x in rows}
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="/content/drive/MyDrive")
    ap.add_argument("--config", default="", help="optional: JSON-Datei mit abweichenden Pfaden")
    ap.add_argument("--skip-gpc", action="store_true")
    ap.add_argument("--skip-articles", action="store_true")
    args, _ = ap.parse_known_args()
    run(args.root, args.config, args.skip_gpc, args.skip_articles)


def run(root, config_path="", skip_gpc=False, skip_articles=False, overrides=None):
    t0 = time.time()
    cfg = dict(DEFAULT_CONFIG)
    if config_path:
        with open(config_path, encoding="utf-8") as fh:
            cfg.update(json.load(fh))
    if overrides:
        cfg.update(overrides)
    out = os.path.join(root, cfg["OUTPUT_DIR"])
    os.makedirs(out, exist_ok=True)
    log(f"Ausgabe: {out}")

    dn = audit_datanorm(root, cfg, out)

    gpc_nums = collections.defaultdict(collections.Counter)
    gpc_rows = [] if skip_gpc else audit_gpc(root, cfg, out, gpc_nums)

    arts, keys = ({}, None) if skip_articles else load_article_list(os.path.join(root, cfg["ARTICLE_LIST"]))
    dncsv = audit_datanorm_csv(os.path.join(root, cfg["DATANORM_CSV"]))

    # Abdeckung
    art_set = set(arts)
    gf_dn = [st for st in dn if st.is_grundfos() and not st.dup_of]
    dn_arts = set().union(*[st.articles for st in gf_dn]) if gf_dn else set()
    dn_ean = set().union(*[st.with_ean for st in gf_dn]) if gf_dn else set()
    dn_price = set().union(*[st.with_price for st in gf_dn]) if gf_dn else set()
    dn_long = set().union(*[st.with_longkey for st in gf_dn]) if gf_dn else set()
    cover = {
        "erstellt": dt.datetime.now().isoformat(timespec="seconds"),
        "laufzeit_min": round((time.time() - t0) / 60, 1),
        "artikelliste": {"anzahl": len(art_set), "felder": keys},
        "datanorm_grundfos_dateien": [
            {"datei": st.label, "datum": st.v_fields()["datum"], "info": st.v_fields()["info"],
             "artikel": len(st.articles), "ean": len(st.with_ean), "preis": len(st.with_price),
             "langtext": len(st.with_longkey)} for st in gf_dn],
        "datanorm_grundfos_gesamt": {
            "artikel_unique": len(dn_arts),
            "davon_in_artikelliste": len(dn_arts & art_set),
            "mit_ean": len(dn_ean),
            "mit_preis": len(dn_price),
            "mit_langtext_schluessel": len(dn_long),
            "nicht_in_artikelliste_beispiele": sorted(dn_arts - art_set)[:25],
        },
        "datanorm_gesamt_csv": {k: v for k, v in dncsv.items() if not k.startswith("_")},
        "gpc_nummern_nach_quelle": {
            src: {"verschiedene": len(c), "davon_in_artikelliste": len(set(c) & art_set),
                  "beispiele": list(c)[:8]}
            for src, c in sorted(gpc_nums.items(), key=lambda kv: -len(kv[1]))[:40]},
    }
    if dncsv.get("_artikel"):
        cover["datanorm_gesamt_csv"]["davon_in_artikelliste"] = len(dncsv["_artikel"] & art_set)
    if arts:
        fam_total = collections.Counter(arts.values())
        fam_dn = collections.Counter(arts[a] for a in dn_arts & art_set)
        cover["abdeckung_nach_familie_top30"] = {
            f: {"artikel": n, "in_datanorm": fam_dn.get(f, 0)} for f, n in fam_total.most_common(30)}
    with open(os.path.join(out, "05_abdeckung.json"), "w", encoding="utf-8") as fh:
        json.dump(cover, fh, ensure_ascii=False, indent=2)

    # Zusammenfassung
    md = ["# Datenaudit – Zusammenfassung", "",
          f"Erstellt: {cover['erstellt']} · Laufzeit: {cover['laufzeit_min']} min", "",
          "## DATANORM", "",
          f"- Dateien analysiert: {len(dn)} (davon Duplikate: {sum(1 for s in dn if s.dup_of)})",
          f"- Grundfos-Dateien (eindeutig): {len(gf_dn)}", ""]
    md.append("| Datei | Hersteller | Datum | Version | Format | Artikel | EAN | Preis | Langtext | Satzarten |")
    md.append("|---|---|---|---|---|---|---|---|---|---|")
    for st in dn:
        if st.dup_of:
            continue
        v = st.v_fields()
        md.append(f"| {st.label} | {st.maker()} | {v['datum']} | {v['version']} | {st.fmt} | "
                  f"{len(st.articles)} | {len(st.with_ean)} | {len(st.with_price)} | {len(st.with_longkey)} | "
                  f"{dict(st.rec)} |")
    g = cover["datanorm_grundfos_gesamt"]
    md += ["", "## Abdeckung Grundfos", "",
           f"- Artikelliste: {len(art_set)} Nummern",
           f"- Grundfos-Artikel in DATANORM: {g['artikel_unique']} (davon in Artikelliste: {g['davon_in_artikelliste']})",
           f"- mit EAN: {g['mit_ean']} · mit Preis: {g['mit_preis']} · mit Langtext-Schlüssel: {g['mit_langtext_schluessel']}",
           "", "## GPC", "", f"- Dateien: {len(gpc_rows)} · Details: `04_gpc_details.md`", ""]
    md.append("| Quelle | verschiedene Grundfos-Nummern | davon in Artikelliste |")
    md.append("|---|---|---|")
    for src, v in cover["gpc_nummern_nach_quelle"].items():
        md.append(f"| {src} | {v['verschiedene']} | {v['davon_in_artikelliste']} |")
    with open(os.path.join(out, "00_zusammenfassung.md"), "w", encoding="utf-8") as fh:
        fh.write("\n".join(md) + "\n")
    log(f"FERTIG in {cover['laufzeit_min']} min → {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/datanorm_import.py
#!/usr/bin/env python3
"""
Grundfos-PIM – Import 1: DATANORM + Artikelliste + Kategoriebaum → Supabase

Ablauf je DATANORM-Datei:
  1. Prüfsumme bilden; bereits importierte Dateien werden übersprungen.
  2. Durchlauf 1: Grundfos-Artikel erkennen (Kurztext enthält GRUNDFOS oder Nummer
     steht in der Artikelliste). Dateien ohne Grundfos werden übersprungen;
     Dateien mit geringem Grundfos-Anteil (Großhandel) werden gefiltert importiert.
  3. Durchlauf 2: Zeilen unverändert nach raw.datanorm_satz, daraus pim.* aufbauen:
     Artikel, EAN, Kurz-/Lang-/Techniktext, Listenpreis, Waren- und Rabattgruppen.

Verbindung: Umgebungsvariable oder Colab-Secret SUPABASE_DB_URL (Session Pooler).
Quelldateien werden nur gelesen.
"""

import csv
import datetime as dt
import hashlib
import io
import json
import os
import re
import time

import psycopg

from datenaudit import GF_NUM, ean_ok, guess_encoding, log, looks_like_datanorm

CONFIG = {
    "DATANORM_DIRS": [
        "04 Datanorm, ENF & Herstellerdaten/Datanorm X Shopify",
        "GRUNDFOS – Zentrale Projektablage/DATANORM NEU 1",
        "05 Rohdaten Import",
    ],
    "ARTICLE_LIST": "02 Grundfos/GRUNDFOS/de_DE/checkpoints/product_list.jsonl",
    "PRODUCT_GROUPS": "02 Grundfos/Grundfos Daten X Shopify/GRUNDFOS_Product_Groups_de_DE_MASTER.csv",
    "VOLL_AB_ANTEIL": 0.5,   # ab diesem Grundfos-Anteil wird die Datei vollständig übernommen
}
HERSTELLER_ID = 1  # pim.hersteller 'Grundfos'
PREISEINHEIT = {"0": 1, "1": 10, "2": 100, "3": 1000}


# ───────────────────────────────────────────────────────────────────
# Verbindung & Hilfen
# ───────────────────────────────────────────────────────────────────
def db_url():
    url = os.environ.get("SUPABASE_DB_URL")
    if not url:
        try:
            from google.colab import userdata  # type: ignore
            url = userdata.get("SUPABASE_DB_URL")
        except Exception:  # noqa: BLE001
            url = None
    if not url:
        raise SystemExit("SUPABASE_DB_URL fehlt (Colab: Schlüssel-Symbol → Secret anlegen).")
    return url.strip()


def conn_params(url):
    """Verbindungszeile robust zerlegen – auch wenn das Passwort Sonderzeichen wie [ ] @ # / ? enthält."""
    m = re.match(r"^postgres(?:ql)?://([^:/@]+):(.*)@([^@/:]+):(\d+)/([^?]*)", url.strip())
    if not m:
        raise SystemExit("Verbindungszeile nicht erkannt. Erwartet: "
                         "postgresql://postgres.<projekt>:<passwort>@<host>.pooler.supabase.com:5432/postgres")
    user, pw, host, port, db = m.groups()
    if pw.startswith("[") and pw.endswith("]"):
        pw = pw[1:-1]          # eckige Klammern aus der Vorlage mitkopiert
    if pw == "YOUR-PASSWORD":
        raise SystemExit("Im Secret steht noch [YOUR-PASSWORD] – bitte durch das Datenbank-Passwort ersetzen.")
    if host.startswith("db."):
        raise SystemExit("Das ist die direkte Verbindung (db.…supabase.co), die Colab nicht erreicht. "
                         "Bitte Supabase → Connect → Direct → „Session pooler“ verwenden.")
    return {"host": host, "port": int(port), "user": user, "password": pw, "dbname": db or "postgres",
            "connect_timeout": 20, "sslmode": "disable" if host in ("localhost", "127.0.0.1") else "require"}


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def norm(nr):
    return (nr or "").strip().upper().replace(" ", "")


def parse_datum(s):
    s = (s or "").strip()
    for fmt in ("%d%m%y", "%d%m%Y", "%d.%m.%Y"):
        try:
            return dt.datetime.strptime(s, fmt).date()
        except ValueError:
            pass
    return None


def v_info(v):
    """Datum und Infotext aus V-Satz (Fest- oder Semikolonformat)."""
    if not v:
        return None, ""
    if len(v) > 1 and v[1] == ";":
        f = [x.strip() for x in v.split(";")]
        datum = next((parse_datum(x) for x in f if re.fullmatch(r"\d{6}|\d{8}", x)), None)
        return datum, " | ".join(x for x in f[1:] if x and not re.fullmatch(r"\d{6}|\d{8}|0?4|EUR", x))
    return parse_datum(v[2:8]), re.sub(r"\s+", " ", v[8:123]).strip()


def read_lines(path):
    with open(path, "rb") as fh:
        head = fh.read(256 * 1024)
    enc = guess_encoding(head)
    with open(path, encoding=enc, errors="replace", newline="") as fh:
        for i, line in enumerate(fh, 1):
            line = line.rstrip("\r\n")
            if line:
                yield i, line


def text_triples(f, start):
    """T/E-Sätze: (Zeilennummer; ; Text) wiederholt ab Feld `start`."""
    out, j = [], start
    while j < len(f):
        tok = f[j].strip()
        if tok.isdigit() and j + 2 < len(f):
            out.append((int(tok), f[j + 2].strip()))
            j += 3
        else:
            j += 1
    return out


def copy_rows(cur, table, cols, rows):
    with cur.copy(f"COPY {table} ({', '.join(cols)}) FROM STDIN") as cp:
        for r in rows:
            cp.write_row(r)


# ───────────────────────────────────────────────────────────────────
# Artikelliste + Kategoriebaum
# ───────────────────────────────────────────────────────────────────
def register_file(cur, lauf_id, path, rel, fmt, lieferant, datenstand, kopf, sprache=None):
    sha = sha256_of(path)
    cur.execute("select id from raw.quelldatei where sha256 = %s", (sha,))
    if cur.fetchone():
        return None
    cur.execute(
        """insert into raw.quelldatei (lauf_id, pfad, sha256, groesse, format, lieferant, datenstand, sprache, kopfzeile)
           values (%s,%s,%s,%s,%s,%s,%s,%s,%s) returning id""",
        (lauf_id, rel, sha, os.path.getsize(path), fmt, lieferant, datenstand, sprache, kopf))
    return cur.fetchone()[0]


def new_quelle(cur, datei_id, typ, bez, datenstand, prio):
    cur.execute("""insert into pim.quelle (datei_id, typ, bezeichnung, datenstand, prioritaet)
                   values (%s,%s,%s,%s,%s) returning id""", (datei_id, typ, bez, datenstand, prio))
    return cur.fetchone()[0]


def import_groups(con, root, lauf_id):
    path = os.path.join(root, CONFIG["PRODUCT_GROUPS"])
    if not os.path.exists(path):
        log(f"⚠ Produktgruppen fehlen: {path}")
        return
    with con.cursor() as cur:
        datei_id = register_file(cur, lauf_id, path, CONFIG["PRODUCT_GROUPS"], "csv", "Grundfos Sitemap",
                                 dt.date(2026, 8, 3), "GRUNDFOS_Product_Groups_de_DE_MASTER")
        if datei_id is None:
            log("Produktgruppen: bereits importiert")
            return
        with open(path, encoding="utf-8-sig", newline="") as fh:
            rows = list(csv.DictReader(fh))
        copy_rows(cur, "raw.tabellen_zeile", ["datei_id", "tabelle", "zeile", "daten"],
                  ((datei_id, "product_groups", i, json.dumps(r, ensure_ascii=False)) for i, r in enumerate(rows, 1)))
        qid = new_quelle(cur, datei_id, "artikelliste", "Grundfos Produktgruppen (Sitemap 08/2026)",
                         dt.date(2026, 8, 3), 30)
        copy_rows(cur, "pim.kategorie", ["code", "ebene", "pfad", "quelle_id"],
                  ((r["group_id"], r.get("hierarchy_level"), r.get("hierarchy_path"), qid) for r in rows))
        cur.execute("""update pim.kategorie k set parent_id = p.id
                       from raw.tabellen_zeile z, pim.kategorie p
                       where z.datei_id = %s and z.daten->>'group_id' = k.code
                         and p.code = z.daten->>'parent_group'""", (datei_id,))
    con.commit()
    log(f"Produktgruppen: {len(rows)} Kategorien")


def import_article_list(con, root, lauf_id):
    path = os.path.join(root, CONFIG["ARTICLE_LIST"])
    if not os.path.exists(path):
        log(f"⚠ Artikelliste fehlt: {path}")
        return
    with con.cursor() as cur:
        datei_id = register_file(cur, lauf_id, path, CONFIG["ARTICLE_LIST"], "jsonl", "Grundfos Sitemap",
                                 dt.date(2026, 8, 3), "product_list.jsonl")
        if datei_id is None:
            log("Artikelliste: bereits importiert")
            return
        qid = new_quelle(cur, datei_id, "artikelliste", "Grundfos Sitemap-Artikelliste 08/2026",
                         dt.date(2026, 8, 3), 30)
        # Platz sparen: die 806.000 JSON-Zeilen nicht zusätzlich als Rohdaten speichern –
        # die Quelldatei ist über Pfad + SHA-256 in raw.quelldatei eindeutig nachvollziehbar.
        cur.execute("""create temp table tmp_liste (artnr text, norm text,
                       familie text, serie text, url text, name text) on commit drop""")
        log("Artikelliste: hochladen …")
        with open(path, encoding="utf-8", errors="replace") as fh, \
                cur.copy("COPY tmp_liste FROM STDIN") as cp:
            for i, line in enumerate(fh, 1):
                line = line.strip()
                if not line:
                    continue
                o = json.loads(line)
                nr = str(o.get("article_number") or "")
                cp.write_row((nr, norm(nr), o.get("family"), o.get("series"),
                              o.get("url"), o.get("product_name")))
        cur.execute("update raw.quelldatei set zeilen = (select count(*) from tmp_liste) where id = %s",
                    (datei_id,))
        cur.execute("""insert into pim.artikel (hersteller_id, artikelnummer, artikelnummer_norm, familie, serie, hersteller_url)
                       select distinct on (norm) %s, artnr, norm, familie, serie, url from tmp_liste where norm <> ''
                       on conflict (hersteller_id, artikelnummer_norm) do update
                         set familie = coalesce(pim.artikel.familie, excluded.familie),
                             serie = coalesce(pim.artikel.serie, excluded.serie),
                             hersteller_url = coalesce(pim.artikel.hersteller_url, excluded.hersteller_url),
                             aktualisiert_am = now()""", (HERSTELLER_ID,))
        cur.execute("""insert into pim.artikel_text (artikel_id, sprache, art, text, quelle_id)
                       select distinct on (a.id) a.id, 'de', 'titel', t.name, %s
                       from tmp_liste t join pim.artikel a on a.hersteller_id = %s and a.artikelnummer_norm = t.norm
                       where coalesce(t.name, '') <> ''
                       on conflict do nothing""", (qid, HERSTELLER_ID))
        cur.execute("""update pim.artikel a set kategorie_id = k.id
                       from pim.kategorie k
                       where a.kategorie_id is null and k.code = a.familie || '/' || a.serie""")
        cur.execute("""update pim.artikel a set kategorie_id = k.id
                       from pim.kategorie k where a.kategorie_id is null and k.code = a.familie""")
    con.commit()
    log("Artikelliste: fertig")


# ───────────────────────────────────────────────────────────────────
# DATANORM
# ───────────────────────────────────────────────────────────────────
def scan_datanorm(path, known):
    """Durchlauf 1: Kopf, Grundfos-Artikel und benötigte Text-Schlüssel."""
    v, n_a, gf = "", 0, set()
    keys = set()
    for _i, line in read_lines(path):
        rt = line[0].upper()
        if rt == "V" and not v:
            v = line[:300]
        if len(line) < 2 or line[1] != ";":
            continue
        f = line.split(";")
        if rt == "A" and len(f) > 5:
            n_a += 1
            nr = norm(f[2])
            if "GRUNDFOS" in (f[4] + f[5]).upper() or nr in known:
                gf.add(nr)
                if len(f) > 12 and f[12].strip():
                    keys.add(f[12].strip())
        elif rt == "D" and len(f) > 5 and norm(f[2]) in gf:
            for j in range(3, len(f) - 3, 4):
                if f[j + 2].strip():
                    keys.add(f[j + 2].strip())
    return v, n_a, gf, keys


def import_datanorm_file(con, root, lauf_id, path, known):
    rel = os.path.relpath(path, root)
    v, n_a, gf, keys = scan_datanorm(path, known)
    if not v:   # Folgedatei (datanorm.002 …) ohne eigenen V-Satz: Kopf aus der .001 im selben Ordner
        cand = os.path.splitext(path)[0] + ".001"
        if cand != path and os.path.exists(cand):
            for _i, line in read_lines(cand):
                if line[:1].upper() == "V":
                    v = line[:300]
                    break
    datum, info = v_info(v)
    is_aux = os.path.splitext(path)[1].lower() in (".wrg", ".rab")
    hersteller_kopf = "GRUNDFOS" in v.upper()
    # Grundfos-Daten über Großhandel/Datendienst; Kopf z. B. „V 161215GF alle Preisangaben in Euro“ (Datum direkt davor)
    gf_code = bool(re.search(r"(^|[;\s\d])GF alle Preisangaben", v))
    gf_kopf = hersteller_kopf or gf_code
    if not gf_kopf:
        # Entscheidung 24.10.2026: nur Grundfos-Quellen (Grundfos GmbH, „GF“-Datei); Großhändler und
        # Fremdhersteller werden nicht übernommen
        log(f"  überspringe (keine Grundfos-Quelle): {rel}")
        return
    anteil = len(gf) / n_a if n_a else 1.0
    voll = anteil >= CONFIG["VOLL_AB_ANTEIL"] or gf_kopf
    with con.cursor() as cur:
        datei_id = register_file(cur, lauf_id, path, rel, "datanorm4", info[:200], datum, v[:300], "de")
        if datei_id is None:
            log(f"  bereits importiert: {rel}")
            return
        if not voll:
            cur.execute("update raw.quelldatei set kopfzeile = kopfzeile || %s where id = %s",
                        (f"  [gefiltert: nur {len(gf)} Grundfos-Artikel von {n_a}]", datei_id))
        typ = "hersteller_datanorm" if hersteller_kopf else "grosshandel_datanorm"
        prio = 80 if hersteller_kopf else (70 if gf_code else 60)
        qid = new_quelle(cur, datei_id, typ, info[:200] or rel, datum, prio)

        raw_rows, art, ean, texte, preise, bausteine = [], {}, {}, [], [], []
        d_vals = {}          # (artnr) -> list of (zeile, ekey, werte)
        wg, rg = [], []
        n = 0
        for i, line in read_lines(path):
            rt = line[0].upper()
            f = line.split(";") if len(line) > 1 and line[1] == ";" else [line]
            nr = norm(f[2]) if len(f) > 2 else ""
            keep = voll or rt in "VSR" or (rt in "ABDP" and nr in gf) or \
                (rt in "TE" and len(f) > 2 and f[2].strip() in keys)
            if not keep:
                continue
            raw_rows.append((datei_id, i, rt, f))
            n += 1
            if len(raw_rows) >= 20000:
                copy_rows(cur, "raw.datanorm_satz", ["datei_id", "zeile", "satzart", "felder"], raw_rows)
                raw_rows = []
            if len(f) < 3:
                continue
            if rt == "A" and nr in gf and len(f) > 9:
                art[nr] = f[2].strip()
                for a, idx in (("kurz1", 4), ("kurz2", 5)):
                    if f[idx].strip():
                        texte.append((nr, a, f[idx].strip()))
                preis = re.sub(r"\D", "", f[9])
                if preis and int(preis) > 0:
                    preise.append((nr, "listenpreis" if f[6].strip() in ("1", "") else "ek",
                                   int(preis) / 100, PREISEINHEIT.get(f[7].strip(), 1), f[8].strip() or None,
                                   f[10].strip() if len(f) > 10 else None, datum))
                if len(f) > 12 and f[12].strip():
                    texte.append((nr, "_langkey", f[12].strip()))
                if len(f) > 11 and f[11].strip():
                    texte.append((nr, "_hwg", f[11].strip()))
            elif rt == "B" and nr in gf:
                e = next((x.strip() for x in f[3:] if ean_ok(x.strip())), None)
                if e:
                    ean[nr] = e
            elif rt in "TE":
                for z, t in text_triples(f, 3):
                    bausteine.append((qid, f[2].strip(), "de", z if rt == "T" else 100000 + z, t))
            elif rt == "D" and nr in gf:
                for j in range(3, len(f) - 3, 4):
                    if f[j].strip().isdigit():
                        d_vals.setdefault(nr, []).append((int(f[j]), f[j + 2].strip(), f[j + 3]))
            elif rt == "S" and len(f) > 5:
                wg.append((f[2].strip() + (("/" + f[4].strip()) if f[4].strip() else ""), f[2].strip(), f[5].strip()))
            elif rt == "R" and len(f) > 5:
                rg.append((f[2].strip(), f[5].strip()))
        if raw_rows:
            copy_rows(cur, "raw.datanorm_satz", ["datei_id", "zeile", "satzart", "felder"], raw_rows)
        cur.execute("update raw.quelldatei set zeilen = %s where id = %s", (n, datei_id))

        # Artikel anlegen / EAN ergänzen
        cur.execute("create temp table tmp_a (nr text, norm text, ean text) on commit drop")
        copy_rows(cur, "tmp_a", ["nr", "norm", "ean"], ((v_, k, ean.get(k)) for k, v_ in art.items()))
        cur.execute("""insert into pim.artikel (hersteller_id, artikelnummer, artikelnummer_norm, ean)
                       select %s, nr, norm, ean from tmp_a
                       on conflict (hersteller_id, artikelnummer_norm) do update
                         set ean = coalesce(pim.artikel.ean, excluded.ean), aktualisiert_am = now()""",
                    (HERSTELLER_ID,))
        cur.execute("create temp table tmp_map on commit drop as select a.id, a.artikelnummer_norm as norm "
                    "from pim.artikel a join tmp_a t on t.norm = a.artikelnummer_norm where a.hersteller_id = %s",
                    (HERSTELLER_ID,))
        cur.execute("create index on tmp_map (norm)")

        # Textbausteine
        if bausteine:
            cur.execute("create temp table tmp_b (quelle_id bigint, schluessel text, sprache text, zeile int, text text) on commit drop")
            copy_rows(cur, "tmp_b", ["quelle_id", "schluessel", "sprache", "zeile", "text"], bausteine)
            cur.execute("""insert into pim.textbaustein select distinct on (quelle_id, schluessel, sprache, zeile) *
                           from tmp_b on conflict do nothing""")

        # Langtext (T-Bausteine) und Techniktext (E-Vorlagen + D-Werte) je Artikel
        bs = {}
        for _q, k, _s, z, t in bausteine:
            bs.setdefault(k, {})[z] = t
        lang_rows = []
        langkey = {nr: t for nr, a, t in texte if a == "_langkey"}
        for nr, key in langkey.items():
            lines = bs.get(key, {})
            txt = " ".join(t for z, t in sorted(lines.items()) if z < 100000 and t)
            if txt:
                lang_rows.append((nr, "lang", fix_hyphen(txt)))
        for nr, vals in d_vals.items():
            parts = []
            for z, ekey, werte in sorted(vals):
                vorlage = bs.get(ekey, {}).get(100000 + z, "")
                parts.append(fill_template(vorlage, werte))
            # E-Zeilen ohne D-Werte (feste Technikzeilen) ergänzen
            ekeys = {e for _z, e, _w in vals}
            used = {z for z, _e, _w in vals}
            for ekey in sorted(ekeys):
                for z, t in sorted(bs.get(ekey, {}).items()):
                    if z >= 100000 and (z - 100000) not in used and "$" not in t:
                        parts.append(t)
            txt = "\n".join(p for p in parts if p.strip())
            if txt:
                lang_rows.append((nr, "technik", txt))
        kurz = [(nr, a, t) for nr, a, t in texte if not a.startswith("_")]
        cur.execute("create temp table tmp_t (norm text, art text, text text) on commit drop")
        copy_rows(cur, "tmp_t", ["norm", "art", "text"], kurz + lang_rows)
        cur.execute("""insert into pim.artikel_text (artikel_id, sprache, art, text, quelle_id)
                       select distinct on (m.id, t.art) m.id, 'de', t.art, t.text, %s
                       from tmp_t t join tmp_map m using (norm) on conflict do nothing""", (qid,))

        # Preise
        cur.execute("""create temp table tmp_p (norm text, preisart text, betrag numeric, preiseinheit int,
                       mengeneinheit text, rabattgruppe text, gueltig_ab date) on commit drop""")
        copy_rows(cur, "tmp_p", ["norm", "preisart", "betrag", "preiseinheit", "mengeneinheit",
                                 "rabattgruppe", "gueltig_ab"], preise)
        cur.execute("""insert into pim.artikel_preis (artikel_id, markt, preisart, betrag, waehrung, preiseinheit,
                         mengeneinheit, rabattgruppe, gueltig_ab, quelle_id)
                       select m.id, 'DE', p.preisart, p.betrag, 'EUR', p.preiseinheit, p.mengeneinheit,
                              p.rabattgruppe, p.gueltig_ab, %s
                       from tmp_p p join tmp_map m using (norm)""", (qid,))

        # Waren- und Rabattgruppen (nur aus Grundfos-Dateien)
        if hersteller_kopf:
            for code, haupt, bez in wg:
                cur.execute("""insert into pim.warengruppe (hersteller_id, code, hauptgruppe, bezeichnung, quelle_id)
                               values (%s,%s,%s,%s,%s) on conflict (hersteller_id, code) do update
                               set bezeichnung = excluded.bezeichnung, quelle_id = excluded.quelle_id""",
                            (HERSTELLER_ID, code, haupt, bez, qid))
            for code, bez in rg:
                cur.execute("""insert into pim.rabattgruppe (hersteller_id, code, bezeichnung, quelle_id)
                               values (%s,%s,%s,%s) on conflict (hersteller_id, code) do update
                               set bezeichnung = excluded.bezeichnung, quelle_id = excluded.quelle_id""",
                            (HERSTELLER_ID, code, bez, qid))
    con.commit()
    log(f"  ✔ {rel}: {n} Zeilen roh, {len(art)} Grundfos-Artikel, {len(ean)} EAN, {len(preise)} Preise, "
        f"{len(lang_rows)} Lang-/Techniktexte ({'voll' if voll else f'gefiltert {anteil:.0%}'})")


def fix_hyphen(txt):
    """Silbentrennung über Zeilengrenzen zusammenziehen: 'Wellen- abdichtung' → 'Wellenabdichtung'."""
    return re.sub(r"(\w)- (?=[a-zäöüß])", r"\1", txt)


def fill_template(vorlage, werte):
    vals = [w for w in werte.split("$")]
    if vals and vals[-1] == "":
        vals.pop()
    if "$" not in vorlage:
        return (vorlage + " " + " ".join(vals)).strip()
    out, k = [], 0
    for ch in vorlage:
        if ch == "$":
            out.append(vals[k] if k < len(vals) else "")
            k += 1
        else:
            out.append(ch)
    return "".join(out)


def import_datanorm(con, root, lauf_id):
    with con.cursor() as cur:
        cur.execute("select artikelnummer_norm from pim.artikel where hersteller_id = %s", (HERSTELLER_ID,))
        known = {r[0] for r in cur.fetchall()}
    log(f"DATANORM: {len(known)} bekannte Grundfos-Nummern zum Abgleich")
    files = []
    for rel in CONFIG["DATANORM_DIRS"]:
        base = os.path.join(root, rel)
        if not os.path.isdir(base):
            log(f"  (Ordner fehlt, übersprungen: {rel})")
            continue
        for dp, _dn, fn in os.walk(base):
            for name in sorted(fn):
                p = os.path.join(dp, name)
                with open(p, "rb") as fh:
                    head = fh.read(4096)
                if looks_like_datanorm(name, head):
                    files.append(p)
    log(f"DATANORM: {len(files)} Dateien gefunden")
    for p in files:
        try:
            import_datanorm_file(con, root, lauf_id, p, known)
        except Exception as e:  # noqa: BLE001 – eine kaputte Datei soll den Lauf nicht stoppen
            con.rollback()
            log(f"  ⚠ Fehler bei {os.path.relpath(p, root)}: {e!r}")


def link_cross_file_langtexts(con):
    """Langtexte, deren T-Sätze in einer anderen Datei gleichen Datenstands liegen (z. B. .001 + .002)."""
    with con.cursor() as cur:
        cur.execute(r"""
            insert into pim.artikel_text (artikel_id, sprache, art, text, quelle_id)
            select a.id, 'de', 'lang',
                   regexp_replace(string_agg(tb.text, ' ' order by tb.zeile), '(\w)- ([a-zäöüß])', '\1\2', 'g'),
                   q.id
            from raw.datanorm_satz s
            join pim.quelle q on q.datei_id = s.datei_id
            join pim.artikel a on a.hersteller_id = %s
                              and a.artikelnummer_norm = upper(replace(trim(s.felder[3]), ' ', ''))
            join pim.quelle q2 on q2.datenstand = q.datenstand and q2.id <> q.id
            join pim.textbaustein tb on tb.quelle_id = q2.id and tb.schluessel = trim(s.felder[13])
                                    and tb.zeile < 100000
            where s.satzart = 'A' and coalesce(trim(s.felder[13]), '') <> ''
              and not exists (select 1 from pim.artikel_text x
                              where x.artikel_id = a.id and x.art = 'lang' and x.quelle_id = q.id)
            group by a.id, q.id
            on conflict do nothing""", (HERSTELLER_ID,))
        n = cur.rowcount
    con.commit()
    log(f"Langtexte aus Nachbardateien verknüpft: {n}")


def run(root="/content/drive/MyDrive", overrides=None):
    if overrides:
        CONFIG.update(overrides)
    t0 = time.time()
    with psycopg.connect(**conn_params(db_url()), autocommit=False) as con:
        with con.cursor() as cur:
            cur.execute("insert into raw.import_lauf (werkzeug, notiz) values (%s, %s) returning id",
                        ("datanorm_import v1", "DATANORM + Artikelliste + Produktgruppen"))
            lauf_id = cur.fetchone()[0]
        con.commit()
        import_groups(con, root, lauf_id)
        import_article_list(con, root, lauf_id)
        import_datanorm(con, root, lauf_id)
        link_cross_file_langtexts(con)
        with con.cursor() as cur:
            cur.execute("update raw.import_lauf set beendet_am = now() where id = %s", (lauf_id,))
            cur.execute("""select (select count(*) from pim.artikel),
                                  (select count(*) from pim.artikel where ean is not null),
                                  (select count(distinct artikel_id) from pim.artikel_preis),
                                  (select count(distinct artikel_id) from pim.artikel_text where art = 'lang'),
                                  (select count(distinct artikel_id) from pim.artikel_text where art = 'technik')""")
            a, e, p, l, t = cur.fetchone()
        con.commit()
    log(f"FERTIG in {(time.time() - t0) / 60:.1f} min · Artikel {a} · mit EAN {e} · mit Preis {p} · "
        f"mit Langtext {l} · mit Techniktext {t}")


if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="/content/drive/MyDrive")
    run(ap.parse_args().root)

In [ ]:
%%writefile /content/projektablage_import.py
#!/usr/bin/env python3
"""
Grundfos-PIM – Import 5: alles Übrige aus „GRUNDFOS – Zentrale Projektablage“ → Supabase (raw)

Bisher fehlten (Prüfung 26.10.2026):
  - Website-Daten 08/2026: GRUNDFOS_Active_Products_de_DE_MASTER.xlsx + 33 Teildateien GRUNDFOS_Active_de_DE_*.xlsx,
    GRUNDFOS_SpareParts_de_DE_MASTER.xlsx, GRUNDFOS_Pump_SparePart_Mapping_de_DE_MASTER.xlsx, Audits/Summary/Index
  - grundfos produkte.xlsx, keywordliste_…grundfos1.xlsx, Grundfos_Quellenabgleich_2015_2016_DATANORM_2026.csv
  - CSV Shopify auch Grundfos (Archiv 2021): First … Ninth Import
  - Bildzuordnungen und Bildquellen (Shopify 2026), VPS-Nachtlauf (OXOMI-Abgleich, Listen, Skripte, Notizen)
  - Original DATANORM 2016 / datanorm.002 (Fortsetzung der „GF“-Datei) → über datanorm_import

Tabellen (CSV/XLSX, auch in ZIPs) werden zeilenweise als JSON gespeichert (raw.tabellen_zeile), Text-Dateien
(md, txt, json, py, sh) als eine Zeile {"text": …}. PDF, EXE und Bilder werden nur verzeichnet (Name, Größe).
Bereits importierte Dateien (gleiche SHA-256) werden übersprungen. Quelldateien werden nur gelesen.
Die Ordner „Historische Grundfos-…“ sind schon mit Import 2–4 erfasst und werden hier ausgelassen.
"""

import hashlib
import io
import json
import os
import sys
import time
import zipfile

import psycopg

import arge_import
import datanorm_import
from datanorm_import import conn_params, db_url
from datenaudit import guess_encoding, log

BASIS = "GRUNDFOS – Zentrale Projektablage"
AUSLASSEN = ("Historische Grundfos-Daten", "Historische Grundfos-Software")
DATANORM_ORDNER = ("DATANORM NEU 1", "DATANORM NEU 2", "DATANORM NEU – Hauptdateien", "Original DATANORM 2016")
TABELLE = {".csv", ".xlsx", ".xlsm"}
TEXT = {".md", ".txt", ".json", ".py", ".sh", ".jsonl"}
TEXT_MAX = 5_000_000
BILD = {".jpg", ".jpeg", ".png", ".gif", ".tif", ".tiff", ".bmp", ".webp"}


def lese(name, data):
    """→ (format, encoding, {tabelle: [zeilen]})"""
    ext = os.path.splitext(name)[1].lower()
    if ext == ".csv":
        enc, rows = arge_import.tabelle_csv(data)
        return "csv", enc, {os.path.basename(name): rows}
    if ext in (".xlsx", ".xlsm"):
        return "xlsx", None, arge_import.tabelle_xlsx(data)
    if ext in TEXT and len(data) <= TEXT_MAX:
        enc = guess_encoding(data[:200000])
        return ext.lstrip("."), enc, {os.path.basename(name): [{"text": data.decode(enc, errors="replace")}]}
    fmt = "bild" if ext in BILD else (ext.lstrip(".") or "datei")   # 'bild' wie ARGE 2015 → Medienzuordnung
    return fmt, None, {"_dateien": [{"name": os.path.basename(name), "groesse": len(data)}]}


def speichere(cur, lauf_id, rel, data, name):
    sha = hashlib.sha256(data).hexdigest()
    cur.execute("select id, zeilen, format from raw.quelldatei where sha256 = %s", (sha,))
    alt = cur.fetchone()
    if alt:
        return None
    fmt, enc, tabellen = lese(name, data)
    cur.execute("""insert into raw.quelldatei (lauf_id, pfad, sha256, groesse, format, lieferant, sprache, kopfzeile)
                   values (%s,%s,%s,%s,%s,%s,%s,%s) returning id""",
                (lauf_id, rel, sha, len(data), fmt, "Projektablage", "de", enc or ""))
    datei_id = cur.fetchone()[0]
    n = 0
    with cur.copy("COPY raw.tabellen_zeile (datei_id, tabelle, zeile, daten) FROM STDIN") as cp:
        for tab, rows in tabellen.items():
            for i, r in enumerate(rows, 1):
                cp.write_row((datei_id, tab, i, json.dumps(r, ensure_ascii=False)))
                n += 1
    cur.execute("update raw.quelldatei set zeilen = %s where id = %s", (n, datei_id))
    return n


def run(root="/content/drive/MyDrive"):
    t0 = time.time()
    base = os.path.join(root, BASIS)
    if not os.path.isdir(base):
        raise SystemExit(f"Ordner „{BASIS}“ nicht gefunden")

    # 1. DATANORM-Ordner der Projektablage (bekannte Dateien werden über die Prüfsumme übersprungen)
    datanorm_import.CONFIG["DATANORM_DIRS"] = [os.path.join(BASIS, d) for d in DATANORM_ORDNER]
    with psycopg.connect(**conn_params(db_url()), autocommit=False) as con:
        with con.cursor() as cur:
            cur.execute("insert into raw.import_lauf (werkzeug, notiz) values (%s, %s) returning id",
                        ("projektablage_import v1", "Zentrale Projektablage: DATANORM-Ordner + übrige Dateien"))
            lauf_id = cur.fetchone()[0]
        con.commit()
        datanorm_import.import_datanorm(con, root, lauf_id)
        datanorm_import.link_cross_file_langtexts(con)
        con.commit()

        # 2. alle übrigen Dateien
        gesamt, neu = 0, 0
        with con.cursor() as cur:
            cur.execute("set statement_timeout = 0")
            for dp, dn, fn in os.walk(base):
                rel_dir = os.path.relpath(dp, base)
                teil = rel_dir.split(os.sep)[0]
                if teil.startswith(AUSLASSEN) or teil in DATANORM_ORDNER:
                    dn[:] = []
                    continue
                for name in sorted(fn):
                    p = os.path.join(dp, name)
                    rel = os.path.relpath(p, root)
                    try:
                        if name.lower().endswith(".zip"):
                            with zipfile.ZipFile(p) as zf:
                                for info in zf.infolist():
                                    if info.is_dir():
                                        continue
                                    n = speichere(cur, lauf_id, rel + "!/" + info.filename, zf.read(info), info.filename)
                                    if n is not None:
                                        neu += 1
                                        gesamt += n
                        else:
                            with open(p, "rb") as fh:
                                data = fh.read()
                            n = speichere(cur, lauf_id, rel, data, name)
                            if n is None:
                                log(f"  bereits vorhanden: {rel}")
                                continue
                            neu += 1
                            gesamt += n
                            log(f"  {rel}: {n:,} Zeilen")
                        con.commit()
                    except Exception as e:  # noqa: BLE001 – eine kaputte Datei soll den Lauf nicht stoppen
                        con.rollback()
                        log(f"  ⚠ Fehler bei {rel}: {e!r}")
            cur.execute("update raw.import_lauf set beendet_am = now() where id = %s", (lauf_id,))
        con.commit()
    log(f"FERTIG in {(time.time() - t0) / 60:.1f} min · {neu} neue Dateien · {gesamt:,} Zeilen")


if __name__ == "__main__":
    run(*sys.argv[1:2])

In [ ]:
%%writefile /content/marktbegleiter_import.py
#!/usr/bin/env python3
"""
Grundfos-PIM – Import 9: Preislisten der Marktbegleiter → Supabase (Schema marktbegleiter, vertraulich)

Ablage in Google Drive – ein Unterordner je Marktbegleiter: „Name (Land)“ oder „domain.tld (Land)“, optional eine
Datei firma.json mit {"name", "domain", "land", "ansprechpartner", "email", "preisart", "preis_typ", "waehrung"}:

    GRUNDFOS – Marktbegleiter Preise/
        Pumpenhandel Muster (DE)/Preisliste_2026-10.xlsx
        haustechnik-beispiel.at (AT)/preise.csv
        Scraping Export (EU)/export.jsonl

Formate: CSV, XLSX/XLSM, JSON/JSONL (strukturierte Scraping-Exporte: Liste von Objekten).
Je Datei:
  1. Original unverändert nach raw.quelldatei/raw.tabellen_zeile; dieselbe Datei (Prüfsumme) wird je Firma nur einmal importiert
  2. Firma über Domain bzw. Name + Land erkennen (marktbegleiter.firma_ermitteln) – keine Dubletten
  3. Spalten erkennen (Hersteller, Artikelnummer/MPN, EAN, Bezeichnung, Baureihe, Kategorie, Preis netto/brutto, MwSt, Währung,
     Preiseinheit, Mindestmenge, Staffel, Versand, Lager, Lieferzeit, gültig ab/bis, URL) → marktbegleiter.import_zeilen
  4. marktbegleiter.import_verarbeiten(): Prüfung, Produkte, Preise (Historie, Dubletten übersprungen), Zuordnung zum Grundfos-Stamm
Ergebnis je Datei und Firma wird ausgegeben (zugeordnet / Vorschläge / nicht zugeordnet / fehlerhaft / Dubletten).
Marktpreise werden nie als eigene Preise übernommen; der Grundfos-Stamm (pim) wird nur gelesen.

Voraussetzung (Colab): Secret SUPABASE_DB_URL. Quelldateien werden nur gelesen.
"""

import datetime as dt
import hashlib
import json
import os
import re
import sys

import psycopg

import projektablage_import
from datanorm_import import conn_params, db_url
from datenaudit import log

ORDNER = "GRUNDFOS – Marktbegleiter Preise"
ENDUNGEN = (".csv", ".xlsx", ".xlsm", ".txt", ".json", ".jsonl")

# Spaltenname (klein) → Feld; Reihenfolge = Vorrang. Preis-Spalten müssen überwiegend Zahlen enthalten.
MUSTER = {
    "hersteller": [r"^hersteller$", r"hersteller(name)?$", r"manufacturer", r"brand", r"marke"],
    "mpn": [r"hersteller.?art", r"herst.?nr", r"\bmpn\b", r"werks.?art", r"grundfos.?(nr|nummer|art)", r"art(ikel)?[ ._-]*n(umme)?r",
            r"artnr", r"material", r"produktnummer", r"product.?(no|number)", r"\bsku\b", r"item.?(no|number)"],
    "ean": [r"\bean\b", r"gtin", r"barcode", r"ean"],
    "bezeichnung": [r"bezeichnung", r"beschreibung", r"artikeltext", r"kurztext", r"produktname", r"titel", r"title", r"name",
                    r"description", r"text"],
    "baureihe": [r"baureihe", r"serie", r"series", r"familie", r"family"],
    "kategorie": [r"kategorie", r"category", r"warengruppe", r"gruppe"],
    "preis_netto": [r"netto", r"\bek\b", r"einkauf", r"net.?price", r"\bnet\b", r"exkl"],
    "preis_brutto": [r"brutto", r"\bvk\b", r"verkauf", r"inkl", r"gross", r"listenpreis", r"\bupe\b", r"\buvp\b", r"preis", r"price"],
    "ust_satz": [r"mwst", r"ust", r"vat", r"steuer"],
    "waehrung": [r"w(ä|ae)hrung", r"currency", r"\bwhg\b"],
    "preiseinheit": [r"preiseinheit", r"\bpe\b", r"price.?unit", r"per\b"],
    "mengeneinheit": [r"mengeneinheit", r"\bme\b", r"einheit", r"unit"],
    "mindestmenge": [r"mindest", r"\bmoq\b", r"min.?(menge|qty|order)"],
    "staffel_ab": [r"staffel", r"ab.?menge", r"tier", r"break"],
    "versand": [r"versand", r"fracht", r"shipping", r"delivery.?cost"],
    "lagerbestand": [r"lager", r"bestand", r"verf(ü|ue)gbar", r"stock", r"availability"],
    "lieferzeit": [r"lieferzeit", r"lead.?time", r"delivery.?time"],
    "gueltig_ab": [r"g(ü|ue)ltig.?(ab|von)", r"valid.?from"],
    "gueltig_bis": [r"g(ü|ue)ltig.?bis", r"valid.?(to|until)"],
    "quelle_url": [r"\burl\b", r"link", r"produktseite"],
}
ZAHLEN = ("preis_netto", "preis_brutto", "ust_satz", "preiseinheit", "mindestmenge", "staffel_ab", "versand")


def zahl(v):
    if v is None:
        return None
    s = re.sub(r"[^\d,.\-]", "", str(v))
    if not s or s in "-.,":
        return None
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".") if s.rfind(",") > s.rfind(".") else s.replace(",", "")
    elif "," in s:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def datum(v):
    if not v:
        return None
    s = str(v).strip()[:10]
    for f in ("%Y-%m-%d", "%d.%m.%Y", "%d/%m/%Y", "%d.%m.%y"):
        try:
            return dt.datetime.strptime(s, f).date()
        except ValueError:
            pass
    return None


def spalten(zeilen):
    """→ {feld: spaltenname}"""
    kopf = list(zeilen[0].keys()) if zeilen else []
    probe = zeilen[:300]
    rollen, belegt = {}, set()
    for feld, muster in MUSTER.items():
        for m in muster:
            treffer = [k for k in kopf if k not in belegt and re.search(m, str(k).lower())]
            if feld in ZAHLEN:   # Preise: überwiegend Zahlen; Nebenspalten (Versand, Staffel …) dürfen lückenhaft sein
                anteil = 0.6 if feld.startswith("preis_") else 0.1
                treffer = [k for k in treffer if sum(zahl(z.get(k)) is not None for z in probe) >= anteil * max(len(probe), 1)]
            if treffer:
                rollen[feld] = treffer[0]
                belegt.add(treffer[0])
                break
    return rollen


def lese_tabelle(name, data):
    ext = os.path.splitext(name)[1].lower()
    if ext in (".json", ".jsonl"):
        text = data.decode("utf-8", errors="replace")
        objs = [json.loads(z) for z in text.splitlines() if z.strip()] if ext == ".jsonl" else json.loads(text)
        if isinstance(objs, dict):
            objs = next((v for v in objs.values() if isinstance(v, list)), [objs])
        return [{k: (json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v) for k, v in o.items()}
                for o in objs if isinstance(o, dict)]
    _fmt, _enc, tabellen = projektablage_import.lese(name, data)
    return max(tabellen.values(), key=len) if tabellen else []    # größtes Blatt


def firma_info(ordnername, pfad):
    info = {}
    fj = os.path.join(pfad, "firma.json")
    if os.path.exists(fj):
        info = json.load(open(fj, encoding="utf-8"))
    m = re.match(r"^(.*?)\s*\(([A-Za-z]{2,3})\)\s*$", ordnername)
    name, land = (m.group(1).strip(), m.group(2).upper()) if m else (ordnername.strip(), None)
    domain = name if re.match(r"^[\w.-]+\.[a-z]{2,}$", name, re.I) else None
    return {"name": info.get("name") or name, "domain": info.get("domain") or domain, "land": info.get("land") or land,
            **{k: info.get(k) for k in ("ansprechpartner", "email", "preisart", "preis_typ", "waehrung")}}


def verarbeite(con, root, pfad, fi):
    rel = os.path.relpath(pfad, root)
    with open(pfad, "rb") as fh:
        data = fh.read()
    sha = hashlib.sha256(data).hexdigest()
    with con.cursor() as cur:
        cur.execute("select marktbegleiter.firma_ermitteln(%s, %s, %s)", (fi["name"], fi["domain"], fi["land"]))
        firma_id = cur.fetchone()[0]
        if fi.get("ansprechpartner") or fi.get("email"):
            cur.execute("""update marktbegleiter.firmen set ansprechpartner = coalesce(%s, ansprechpartner),
                           email = coalesce(%s, email), geaendert_am = now() where id = %s""",
                        (fi.get("ansprechpartner"), fi.get("email"), firma_id))
        cur.execute("select id, status from marktbegleiter.importe where firma_id = %s and datei_sha256 = %s", (firma_id, sha))
        alt = cur.fetchone()
        if alt:
            con.commit()
            log(f"  schon importiert (Import {alt[0]}, {alt[1]}): {rel}")
            return
        # Original sichern (raw; gleiche Datei aus anderem Ordner wird dort nicht doppelt abgelegt)
        cur.execute("insert into raw.import_lauf (werkzeug, notiz) values ('marktbegleiter_import v2', %s) returning id", (rel,))
        lauf_id = cur.fetchone()[0]
        projektablage_import.speichere(cur, lauf_id, rel, data, os.path.basename(pfad))
        cur.execute("select id from raw.quelldatei where sha256 = %s", (sha,))
        datei_id = cur.fetchone()[0]

        zeilen = lese_tabelle(os.path.basename(pfad), data)
        rollen = spalten(zeilen)
        preis_typ = fi.get("preis_typ") or ("netto" if "preis_netto" in rollen and "preis_brutto" not in rollen else "brutto")
        preis_sp = rollen.get("preis_" + preis_typ) or rollen.get("preis_brutto") or rollen.get("preis_netto")
        if preis_sp == rollen.get("preis_netto") and preis_sp != rollen.get("preis_brutto"):
            preis_typ = "netto"
        stand = dt.date.fromtimestamp(os.path.getmtime(pfad))
        cur.execute("""insert into marktbegleiter.importe (firma_id, dateiname, datei_sha256, datei_id, format, quelle, preisart,
                                                          stand, spalten)
                       values (%s, %s, %s, %s, %s, %s, %s, %s, %s) returning id""",
                    (firma_id, os.path.basename(pfad), sha, datei_id, os.path.splitext(pfad)[1].lstrip(".").lower(),
                     "scraping" if pfad.lower().endswith((".json", ".jsonl")) else "preisliste",
                     fi.get("preisart") or "haendlerpreis", stand, json.dumps({**rollen, "preis": preis_sp, "preis_typ": preis_typ},
                                                                               ensure_ascii=False)))
        import_id = cur.fetchone()[0]
        g = lambda z, f: z.get(rollen[f]) if f in rollen else None   # noqa: E731
        with cur.copy("""COPY marktbegleiter.import_zeilen (import_id, zeile, rohdaten, hersteller, mpn, ean, bezeichnung, baureihe,
                          kategorie, url, preis, preis_typ, ust_satz, waehrung, preiseinheit, mengeneinheit, mindestmenge, staffel_ab,
                          versand, lagerbestand, lieferzeit, gueltig_ab, gueltig_bis, erfasst_am, quelle_url, land) FROM STDIN""") as cp:
            for i, z in enumerate(zeilen, 1):
                whg = (str(g(z, "waehrung") or "").strip().upper()[:3] or fi.get("waehrung") or "EUR")
                cp.write_row((import_id, i, json.dumps(z, ensure_ascii=False), g(z, "hersteller"),
                              str(g(z, "mpn")).strip() if g(z, "mpn") not in (None, "") else None,
                              str(g(z, "ean")).strip() if g(z, "ean") not in (None, "") else None,
                              g(z, "bezeichnung"), g(z, "baureihe"), g(z, "kategorie"), g(z, "quelle_url"),
                              zahl(z.get(preis_sp)) if preis_sp else None, preis_typ, zahl(g(z, "ust_satz")), whg,
                              zahl(g(z, "preiseinheit")), g(z, "mengeneinheit"), zahl(g(z, "mindestmenge")), zahl(g(z, "staffel_ab")),
                              zahl(g(z, "versand")), g(z, "lagerbestand"), g(z, "lieferzeit"), datum(g(z, "gueltig_ab")),
                              datum(g(z, "gueltig_bis")), stand, g(z, "quelle_url"), fi["land"]))
        cur.execute("select schritt, anzahl from marktbegleiter.import_verarbeiten(%s)", (import_id,))
        erg = dict(cur.fetchall())
    con.commit()
    log(f"  {rel}: {erg} · Spalten {rollen}")


def _ordnername(n):
    """Vergleichsform: Groß/klein, Bindestrich-Varianten (– — -) und Leerzeichen egal."""
    return re.sub(r"[\s\-–—_]+", " ", n).strip().lower()


def finde_ordner(root, ordner=ORDNER):
    """Ordner in „Meine Ablage“ suchen – direkt oder bis zu zwei Ebenen tiefer (z. B. GRUNDFOS/…), auch geteilte Ablagen."""
    ziel = _ordnername(ordner)
    wurzeln = [root] + [d for d in ("/content/drive/Shareddrives", "/content/drive/Shared drives") if os.path.isdir(d)]
    aehnlich = []
    for w in wurzeln:
        for dp, dn, _fn in os.walk(w):
            tiefe = os.path.relpath(dp, w).count(os.sep) + (0 if dp == w else 1)
            for d in dn:
                if _ordnername(d) == ziel:
                    return os.path.join(dp, d)
                if "marktbegleiter" in d.lower():
                    aehnlich.append(os.path.join(dp, d))
            if tiefe >= 2:
                dn[:] = []
            else:
                dn[:] = [d for d in dn if not d.startswith(".")]
    passend = [a for a in aehnlich if "preis" in os.path.basename(a).lower()]
    if len(passend) == 1:
        return passend[0]
    hinweis = ("\n  Gefundene Ordner mit „Marktbegleiter“ im Namen:\n    " + "\n    ".join(aehnlich)) if aehnlich else ""
    raise SystemExit(f"Ordner „{ordner}“ nicht gefunden – bitte in Google Drive unter „Meine Ablage“ anlegen, "
                     f"darin je Marktbegleiter ein Unterordner „Name (Land)“ mit den Preislisten.{hinweis}")


def run(root="/content/drive/MyDrive", ordner=ORDNER):
    base = finde_ordner(root, ordner)
    log(f"Ordner: {base}")
    lose = [f for f in os.listdir(base) if os.path.isfile(os.path.join(base, f)) and f.lower().endswith(ENDUNGEN)]
    if lose:
        log(f"  ⚠ {len(lose)} Datei(en) liegen direkt im Ordner und werden übersprungen – bitte in den Unterordner "
            f"der jeweiligen Firma „Name (Land)“ verschieben: {', '.join(sorted(lose)[:10])}")
    with psycopg.connect(**conn_params(db_url()), autocommit=False) as con:
        with con.cursor() as cur:
            cur.execute("set statement_timeout = 0")
        dateien = 0
        for eintrag in sorted(os.listdir(base)):
            sub = os.path.join(base, eintrag)
            if not os.path.isdir(sub):
                continue
            fi = firma_info(eintrag, sub)
            log(f"{fi['name']} ({fi['land'] or 'Land?'})")
            if not fi["land"]:
                log("  ⚠ kein Land im Ordnernamen – bitte „Name (DE)“ o. ä. verwenden")
            for dp, _dn, fn in os.walk(sub):
                for f in sorted(fn):
                    if f.lower().endswith(ENDUNGEN) and f != "firma.json":
                        dateien += 1
                        try:
                            verarbeite(con, root, os.path.join(dp, f), fi)
                        except Exception as e:  # noqa: BLE001 – eine kaputte Datei soll den Lauf nicht stoppen
                            con.rollback()
                            log(f"  ⚠ Fehler bei {f}: {e!r}")
        if not dateien:
            log("  ⚠ keine Preislisten gefunden – je Marktbegleiter einen Unterordner „Name (Land)“ anlegen und die Dateien "
                f"({', '.join(ENDUNGEN)}) hineinlegen")
        with con.cursor() as cur:
            cur.execute("""select f.name, f.land, i.dateiname, i.status, i.positionen, i.zugeordnet, i.vorschlaege,
                                  i.nicht_zugeordnet, i.fehlerhaft, i.dubletten
                             from marktbegleiter.importe i join marktbegleiter.firmen f on f.id = i.firma_id
                            where i.importiert_am > now() - interval '1 day' order by f.name, i.id""")
            log("Ergebnis je Firma (letzte 24 h): Firma · Land · Datei · Status · Positionen · zugeordnet · Vorschläge · "
                "nicht zugeordnet · fehlerhaft · Dubletten")
            for r in cur.fetchall():
                log("  " + " · ".join(str(x) for x in r))
    log("FERTIG")


if __name__ == "__main__":
    run(*sys.argv[1:2])

In [ ]:
%%writefile /content/shop_exporte_import.py
#!/usr/bin/env python3
"""
Grundfos-PIM – Import 10: Shop-Exporte (Scraper) aus Google Drive → Supabase (Schema marktbegleiter, vertraulich)

Quelle: die in „Marktbegleiter aus Drive“ angehakten Shops (Auswahl 27.10.2026). Je Shop eine Exportdatei im
Scraper-Standardformat (source_url, shop_name, manufacturer, brand, product_name, manufacturer_article_number,
shop_article_number, sku, ean, gtin, price, currency, vat_info, availability, delivery_time, product_url, scraped_at …).
Andere Formate (eigene Spaltennamen) laufen über die Spaltenerkennung von Import 9.

Je Datei:
  1. per Drive-ID herunterladen (nur lesen), Prüfsumme – dieselbe Datei wird je Firma nur einmal importiert
  2. Firma über Domain der Produkt-URLs bzw. Name + Land (marktbegleiter.firma_ermitteln)
  3. Zeilen streamen und filtern (FILTER): "alle" = gesamtes Sortiment (Standard), "pumpen" = nur Produkte von
     Pumpenherstellern, "grundfos" = nur Grundfos
  4. Zuordnung der Spalten: Hersteller = manufacturer/brand (sonst aus dem Produktnamen), Artikelnummer = NUR
     manufacturer_article_number (nie Shop-Nummer/SKU), EAN = ean/gtin, Preis netto/brutto aus vat_info,
     Datum = scraped_at → marktbegleiter.import_zeilen → marktbegleiter.import_verarbeiten()
Das Original bleibt in Drive; gespeichert werden Produkte und Preise. Aus der Zwischenablage (import_zeilen) werden die
fehlerfrei verarbeiteten Zeilen danach entfernt – sie stehen vollständig in preise; Fehler und Dubletten bleiben zur Kontrolle.
Marktpreise werden nie als eigene Preise übernommen; der Grundfos-Stamm (pim) wird nur gelesen.

Voraussetzung (Colab): Secret SUPABASE_DB_URL, Drive-Anmeldung (auth.authenticate_user).
"""

import csv
import datetime as dt
import hashlib
import itertools
import json
import os
import re
import sys
from urllib.parse import urlparse

import psycopg

import marktbegleiter_import as mbi
from datanorm_import import conn_params, db_url
from datenaudit import log

FILTER = "alle"            # "alle" (gesamtes Sortiment, Auswahl 27.10.2026) | "pumpen" | "grundfos"
ZIEL = "/content/shopexporte"

# (Schlüssel, Name, Domain, Land, Drive-Datei-ID, Dateiname) – angehakt in „Marktbegleiter aus Drive“
SHOPS = [
    ("1a_pumpen", "1A-Pumpen", "1a-pumpen.de", "DE", "1SlE96gJ30KgsJKy__cjgXtHjOw-3jBQC", "1a_pumpen_products_complete.csv"),
    ("ar_store_shop", "ar-storeshop.com", "ar-storeshop.com", "IT", "1FNDWrMwcsBhAjoBejrNYVd4BpYqKMkaI", "ar_store_shop_products_complete.csv"),
    ("baumarkteu", "BaumarktEU", "baumarkteu.de", "DE", "1KVupWBR4lxeE37LaNE2uZvyjBwjMZX-V", "baumarkteu_products_complete.csv"),
    ("esc_grossiste", "ESC Grossiste", "esc-grossiste.fr", "FR", "1m-zt4wb1yunn1Tz-e0yyNmAfXYgu7h-6", "esc_grossiste_products_complete.csv"),
    ("glo24", "glo24.de", "glo24.de", "DE", "1-gy7z8RT4r6p3aWlxf8kcQIU8qeIUf-l", "glo24_products_complete.csv"),
    ("heima24", "heima24", "heima24.de", "DE", "1wNRfOD6IQGgTqGlJ1-QBrdSY4R0u-oiR", "heima24_products_complete.csv"),
    ("heizung_billiger", "heizung-billiger.de", "heizung-billiger.de", "DE", "11duwuHRS0oS7BqH0C6ZYLoBYJ1NNJ4Z2", "heizung_billiger_products_complete.csv"),
    ("heizungsdiscount24", "heizungsdiscount24", "heizungsdiscount24.de", "DE", "1S_wtWOBDHQjBim42BZ6qS4BdFJGa0PbK", "heizungsdiscount24_products_complete.csv"),
    ("loebbeshop", "Loebbeshop", "loebbeshop.de", "DE", "1esM6Cpy497Xz-kk5nywmOXKM2wxMGwlc", "loebbeshop_products_complete.csv"),
    ("profluid", "Profluid", "", "", "1CzVDiRNZHgoeaLEghQC0Xs33GOPQhN_D", "profluid_products_complete (1).csv"),
    ("promoclima_it", "Promoclima", "promoclima.it", "IT", "1HWjJfMw4kfi6HSgp-g626rDe-ez0In8C", "promoclima_it_full_products.csv"),
    ("psw_shop_fast", "PSW-Shop", "psw-shop.de", "DE", "1KmFbxqkOu02T1ohwhZxHe4TfhqDAYLOV", "psw_shop_fast_products.csv"),
    ("pumpalia", "Pumpalia", "pumpalia.com", "ES", "1A98CuAowATfvvYfGUkm-stFqvQLCcHlD", "pumpalia_products_complete.csv"),
    ("pumpen24_b2c_de", "pumpen24.de", "pumpen24.de", "DE", "1iA8OspUJEkf5b03sqZut-sD3A7a3RhpL", "pumpen24_b2c_de_products_complete.csv"),
    ("pumpen_grosshandel", "pumpen-grosshandel.de", "pumpen-grosshandel.de", "DE", "1yhCPX8IluQ7VaH9W7WQVusXnwNc0jMSM", "pumpen_grosshandel_products_complete (1).csv"),
    ("pumpen_guru", "pumpen-guru.de", "pumpen-guru.de", "DE", "1BeTtU0i9U1i4C1Risr9rFnPZRq6rWExN", "pumpen_guru_products_complete.csv"),
    ("pumpen_heizung", "pumpen-heizung.de", "pumpen-heizung.de", "DE", "1JFBz3EvS0qgpSufx8xpO2ZYuIAcnqKg0", "pumpen_heizung_products.csv"),
    ("pumpendiscounter", "Pumpendiscounter (Reiga GbR)", "pumpendiscounter.de", "DE", "1VR_k03RV9xcARZD2OAV7SOldR5GizD0s", "pumpendiscounter_products_complete.csv"),
    ("pumpengermany", "pumpengermany.de", "pumpengermany.de", "DE", "1iSxgMvkpFay_OgFiS-5aVcTrrn_69qli", "pumpengermany_products_complete.csv"),
    ("salvador_escoda", "Salvador Escoda", "salvadorescoda.com", "ES", "1JdXTFNr0Jxfv1Bt4WDGSjzm1CLy9ZY6g", "salvador_escoda_products_batch_003.csv"),
    ("sanundo", "Sanundo", "sanundo.de", "DE", "1caaJtyzGp60dZnHZhNDe5Z9YSMczhBM4", "sanundo_products_complete.csv"),
    ("sealing", "Sealing", "sealing.com.ua", "UA", "1gnzj1BnHzjSWJ_AKMexv7YLoI10M8OUz", "sealing_products_complete.csv"),
    ("selfio", "Selfio", "selfio.de", "DE", "1YvueanMM6Ha44m8amqmg-xsfGMHv3PSL", "1selfio_products_titel_hersteller_artikelnummer_ean_preis.csv"),
    ("shkhelden", "shkhelden.de", "shkhelden.de", "DE", "1X47NOOVa_Qllxun1RwlOfHCvGrudZ1HQ", "shkhelden_products_complete.csv"),
    ("st_shop24", "st-shop24.de", "st-shop24.de", "DE", "1M6A1-xqnQtItTfYuc82cn635nd5qKkDD", "st_shop24_products_complete.csv"),
    ("tem_srl", "TEM Srl", "temsrl.it", "IT", "1ak8JYMc3KQwhIu2JycUbh705Y5rm1izo", "tem_srl_products_complete.csv"),
    ("tga_shop", "TGA-Shop", "", "DE", "1VKlAj-3yI2nPYyk_NmLloQfyu699iDBd", "tga_shop_products_complete.csv"),
    ("weinmann_schanz", "Weinmann & Schanz", "weinmann-schanz.de", "DE", "1lHigBOXqZu7h59qT0badG2qua-zvZOlY", "weinmann_schanz_products.csv"),
    ("wida_handel", "WIDA Handel", "", "DE", "1XRRF6g_ULcT3OCxkU0p0ovvA4nfdmF1f", "wida_handel_products_complete.csv"),
]

# Pumpenhersteller (Anzeige-Name → Suchmuster); erster Treffer im Produktnamen gilt, wenn manufacturer/brand leer ist
PUMPEN = [
    ("Grundfos", r"grundfos"), ("Wilo", r"\bwilo\b"), ("KSB", r"\bksb\b"), ("DAB", r"\bdab\b"), ("Lowara", r"lowara"),
    ("Xylem", r"xylem"), ("Flygt", r"\bflygt\b"), ("Ebara", r"\bebara\b"), ("Pedrollo", r"pedrollo"), ("Calpeda", r"calpeda"),
    ("Speck", r"\bspeck\b"), ("Biral", r"\bbiral\b"), ("Homa", r"\bhoma\b"), ("Jung Pumpen", r"jung\s*pumpen|pentair\s*jung"),
    ("Zehnder Pumpen", r"zehnder\s*pumpen"), ("Salmson", r"salmson"), ("Espa", r"\bespa\b"), ("Sulzer", r"\bsulzer\b"),
    ("Pentair", r"pentair"), ("Kessel", r"\bkessel\b"), ("SFA", r"\bsfa\b|sanibroy|sanifos|saniflo"), ("Caprari", r"caprari"),
    ("Saer", r"\bsaer\b"), ("Nocchi", r"nocchi"), ("Rovatti", r"rovatti"), ("Zenit", r"\bzenit\b"), ("Tsurumi", r"tsurumi"),
    ("Pentax", r"\bpentax\b"), ("Leo", r"\bleo\s+pump"), ("Smedegaard", r"smedegaard"), ("Taco", r"\btaco\b"),
    ("Deutsche Vortex", r"vortex"), ("Laing", r"\blaing\b"), ("Halm", r"\bhalm\b"), ("T.I.P.", r"\bt\.?i\.?p\.?\b"),
    ("Sterling", r"sterling\s*sihi|\bsihi\b"), ("Oase", r"\boase\b"), ("Gardena", r"gardena"), ("Einhell", r"einhell"),
    ("Metabo", r"metabo"), ("Kärcher", r"k(ä|ae)rcher"), ("Aquatechnik", r"aquatechnik"), ("Elpumps", r"elpumps"),
    ("Ferroli", r"\bferroli\b"), ("Lowara/Vogel", r"\bvogel\b"),
]
PUMPEN_RE = [(n, re.compile(m, re.I)) for n, m in PUMPEN]
GF_NR = re.compile(r"(?<![\d.,])(9\d{7}|[1-9]\d{7})(?![\d.,])")
BRUTTO = re.compile(r"inkl|incl|inclus|incluid|ttc|iva\s*inc|mit\s*mwst|gross|brutto", re.I)
NETTO = re.compile(r"exkl|excl|zzgl|zuzgl|\bht\b|esclus|sin\s*iva|ohne\s*mwst|netto|\bnet\b|plus\s*vat", re.I)
LAENDER = {"germany": "DE", "deutschland": "DE", "austria": "AT", "österreich": "AT", "switzerland": "CH", "schweiz": "CH",
           "france": "FR", "italy": "IT", "italia": "IT", "spain": "ES", "españa": "ES", "netherlands": "NL", "belgium": "BE",
           "poland": "PL", "czech republic": "CZ", "czechia": "CZ", "romania": "RO", "ukraine": "UA", "united kingdom": "GB"}


SCHEMA_LAGER = {"instock": "in stock", "outofstock": "out of stock", "soldout": "ausverkauft", "discontinued": "nicht lieferbar",
                "preorder": "vorbestellbar", "backorder": "nachbestellt", "limitedavailability": "available, begrenzt",
                "instoreonly": "nur im Laden"}


def lager(v):
    """schema.org-Werte (http://schema.org/InStock) lesbar machen, damit „lieferbar“ erkannt wird."""
    if not v:
        return None
    k = re.sub(r"^https?://schema\.org/", "", v.strip(), flags=re.I)
    return SCHEMA_LAGER.get(k.lower(), k)


LEER = {"", "-", "--", "n/a", "na", "none", "null", "not_found", "not found", "unknown", "unbekannt", "k.a.", "0"}


def name_aus_url(url):
    """Produktname aus der URL, wenn der Export keinen hat: …/12027-grundfos-pompa-cm-a-3-4-cod-92889472.html"""
    if not url:
        return None
    teil = urlparse(url).path.rstrip("/").rsplit("/", 1)[-1]
    teil = re.sub(r"\.(html?|php|aspx?)$", "", teil, flags=re.I)
    teil = re.sub(r"^\d+[-_]", "", teil)
    teil = re.sub(r"[-_+]+", " ", teil).strip()
    return teil or None


def pumpenhersteller(hersteller, name):
    """→ Anzeige-Name des Pumpenherstellers oder None"""
    for txt in (hersteller, name):
        if txt:
            for n, rx in PUMPEN_RE:
                if rx.search(txt):
                    return n
    return None


def lade(file_id, datei):
    """Datei per Drive-API (nur lesen) nach ZIEL laden; vorhandene Kopie wird wiederverwendet."""
    os.makedirs(ZIEL, exist_ok=True)
    pfad = os.path.join(ZIEL, re.sub(r"[^\w.() -]", "_", datei))
    if os.path.exists(pfad) and os.path.getsize(pfad) > 0:
        return pfad
    from googleapiclient.discovery import build  # type: ignore
    from googleapiclient.http import MediaIoBaseDownload  # type: ignore
    svc = build("drive", "v3", cache_discovery=False)
    req = svc.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(pfad + ".part", "wb") as fh:
        dl, fertig = MediaIoBaseDownload(fh, req, chunksize=64 * 1024 * 1024), False
        while not fertig:
            _status, fertig = dl.next_chunk()
    os.replace(pfad + ".part", pfad)
    return pfad


def zeilen(pfad):
    """CSV streamen (Trennzeichen erkennen, sehr lange Felder erlauben, NUL-Zeichen entfernen – PostgreSQL lehnt sie ab)."""
    csv.field_size_limit(sys.maxsize)
    fh = open(pfad, encoding="utf-8-sig", errors="replace", newline="")
    probe = fh.read(20000)
    fh.seek(0)
    trenner = max((",", ";", "\t"), key=lambda t: probe.split("\n", 1)[0].count(t))
    return csv.DictReader((z.replace("\x00", "") for z in fh), delimiter=trenner)


def zuordnen(z, standard, generisch):
    """Eine Zeile → Felder für import_zeilen (Scraper-Standard oder Spaltenerkennung von Import 9)."""
    if standard:
        g = lambda *ks: next((z[k].strip() for k in ks if (z.get(k) or "").strip().lower() not in LEER), None)  # noqa: E731
        vat = g("vat_info") or ""
        url = g("product_url", "canonical_url")
        return {
            "hersteller": g("manufacturer", "brand"), "mpn": g("manufacturer_article_number"), "ean": g("ean", "gtin"),
            "bezeichnung": g("product_full_name", "product_name") or name_aus_url(url), "baureihe": g("product_series"),
            "kategorie": g("product_category", "category_path", "breadcrumb"), "url": url,
            # price / price_gross (brutto) oder nur price_net (netto, z. B. sealing.com.ua)
            "preis": mbi.zahl(g("price", "price_gross") or g("price_net")),
            "preis_typ": "netto" if (not g("price", "price_gross") and g("price_net"))
                         or (NETTO.search(vat) and not BRUTTO.search(vat)) else "brutto",
            "waehrung": (g("currency") or "EUR")[:3].upper(),
            "lagerbestand": lager(g("availability", "stock_status")),
            "lieferzeit": g("delivery_time"), "erfasst_am": mbi.datum(g("scraped_at")),
            "land": LAENDER.get((g("source_country") or "").lower()),
            "shop": g("shop_name", "supplier"), "shop_nr": g("shop_article_number", "sku", "article_number"),
        }
    r = generisch
    w = lambda f: (z.get(r[f]) or "").strip() or None if f in r else None  # noqa: E731
    return {"hersteller": w("hersteller"), "mpn": w("mpn"), "ean": w("ean"), "bezeichnung": w("bezeichnung"),
            "baureihe": w("baureihe"), "kategorie": w("kategorie"), "url": w("quelle_url"),
            "preis": mbi.zahl(w("preis_brutto") or w("preis_netto")),
            "preis_typ": "netto" if "preis_netto" in r and "preis_brutto" not in r else "brutto",
            "waehrung": (w("waehrung") or "EUR")[:3].upper(), "lagerbestand": w("lagerbestand"), "lieferzeit": w("lieferzeit"),
            "erfasst_am": None, "land": None, "shop": None, "shop_nr": None}


def verarbeite(con, shop):
    key, name, domain, land, file_id, datei = shop
    pfad = lade(file_id, datei)
    sha = hashlib.sha256()
    with open(pfad, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            sha.update(block)
    sha = sha.hexdigest()

    rd = zeilen(pfad)
    kopf = rd.fieldnames or []
    standard = ("product_url" in kopf and ({"price", "price_gross", "price_net"} & set(kopf))
                and ("manufacturer_article_number" in kopf or "manufacturer" in kopf))
    generisch = {} if standard else mbi.spalten(list(itertools.islice(zeilen(pfad), 300)))

    auszug, gelesen, domains, laender, staende = [], 0, {}, {}, []
    for z in rd:
        gelesen += 1
        f = zuordnen(z, standard, generisch)
        ph = pumpenhersteller(f["hersteller"], f["bezeichnung"])
        if FILTER == "pumpen" and not ph:
            continue
        if FILTER == "grundfos" and ph != "Grundfos":
            continue
        if not f["hersteller"] and ph:
            f["hersteller"] = ph
        if not f["mpn"] and ph == "Grundfos":       # Grundfos-Nummer steht oft nur im Namen, in der URL oder als Shop-Nummer
            for quelle in (f["bezeichnung"], f["url"] and urlparse(f["url"]).path, f.get("shop_nr")):
                m = GF_NR.search(quelle or "")
                if m:
                    f["mpn"] = m.group(1)
                    break
        if f["url"]:
            h = urlparse(f["url"]).hostname or ""
            domains[h] = domains.get(h, 0) + 1
        if f["land"]:
            laender[f["land"]] = laender.get(f["land"], 0) + 1
        if f["erfasst_am"]:
            staende.append(f["erfasst_am"])
        auszug.append((z, f))

    dom = domain or (max(domains, key=domains.get) if domains else None)
    lnd = land or (max(laender, key=laender.get) if laender else "DE")
    stand = max(staende) if staende else dt.date.fromtimestamp(os.path.getmtime(pfad))
    if not auszug:
        log(f"  {name}: {gelesen} Zeilen gelesen, keine passenden Produkte (Filter {FILTER}) – übersprungen")
        return

    with con.cursor() as cur:
        cur.execute("select marktbegleiter.firma_ermitteln(%s, %s, %s)", (name, dom, lnd))
        firma_id = cur.fetchone()[0]
        cur.execute("""select i.id, i.status, (select count(*) from marktbegleiter.preise p where p.import_id = i.id)
                         from marktbegleiter.importe i where i.firma_id = %s and i.datei_sha256 = %s""", (firma_id, sha))
        alt = cur.fetchone()
        if alt and alt[2] == 0:      # früherer Lauf ohne einen einzigen Preis (z. B. Spalten nicht erkannt) → neu einlesen
            cur.execute("delete from marktbegleiter.import_zeilen where import_id = %s", (alt[0],))
            cur.execute("delete from marktbegleiter.importe where id = %s", (alt[0],))
            log(f"  {name}: früherer Import {alt[0]} ohne Preise – wird neu eingelesen")
            alt = None
        if alt:
            con.commit()
            log(f"  {name}: schon importiert (Import {alt[0]}, {alt[1]})")
            return
        cur.execute("""insert into marktbegleiter.importe (firma_id, dateiname, datei_sha256, format, quelle, preisart, stand,
                                                          spalten, notiz)
                       values (%s, %s, %s, 'csv', 'scraping', 'oeffentlich', %s, %s, %s) returning id""",
                    (firma_id, datei, sha, stand,
                     json.dumps({"format": "scraper-standard" if standard else "erkannt", "filter": FILTER,
                                 "zeilen_gelesen": gelesen, "zeilen_uebernommen": len(auszug),
                                 **({} if standard else generisch)}, ensure_ascii=False),
                     f"Drive-Datei {file_id}; Import 10 (Shop-Exporte)"))
        import_id = cur.fetchone()[0]
        with cur.copy("""COPY marktbegleiter.import_zeilen (import_id, zeile, rohdaten, hersteller, mpn, ean, bezeichnung,
                          baureihe, kategorie, url, preis, preis_typ, waehrung, lagerbestand, lieferzeit, erfasst_am,
                          quelle_url, land) FROM STDIN""") as cp:
            for i, (_z, f) in enumerate(auszug, 1):
                roh = {k: f[k] for k in ("shop", "shop_nr", "hersteller", "mpn", "ean", "bezeichnung", "url") if f.get(k)}
                roh.update({"preis": f["preis"], "waehrung": f["waehrung"], "preis_typ": f["preis_typ"]})
                cp.write_row((import_id, i, json.dumps(roh, ensure_ascii=False, default=str), f["hersteller"], f["mpn"],
                              f["ean"], f["bezeichnung"], f["baureihe"], f["kategorie"], f["url"], f["preis"], f["preis_typ"],
                              f["waehrung"], f["lagerbestand"], f["lieferzeit"], f["erfasst_am"] or stand, f["url"], lnd))
        cur.execute("select schritt, anzahl from marktbegleiter.import_verarbeiten(%s)", (import_id,))
        erg = dict(cur.fetchall())
        cur.execute("delete from marktbegleiter.import_zeilen where import_id = %s and status = 'ok'", (import_id,))
    con.commit()
    log(f"  {name} ({dom}, {lnd}): {gelesen} gelesen → {len(auszug)} übernommen · {erg}")


def run(nur=None):
    with psycopg.connect(**conn_params(db_url()), autocommit=False) as con:
        with con.cursor() as cur:
            cur.execute("set statement_timeout = 0")
        con.commit()
        for shop in SHOPS:
            if nur and shop[0] not in nur:
                continue
            log(f"{shop[1]} – {shop[5]}")
            try:
                verarbeite(con, shop)
            except Exception as e:  # noqa: BLE001 – ein kaputter Export soll den Lauf nicht stoppen
                con.rollback()
                log(f"  ⚠ Fehler bei {shop[1]}: {e!r}")
        with con.cursor() as cur:
            cur.execute("""select f.name, f.land, i.status, i.positionen, i.zugeordnet, i.vorschlaege, i.nicht_zugeordnet,
                                  i.fehlerhaft, i.dubletten
                             from marktbegleiter.importe i join marktbegleiter.firmen f on f.id = i.firma_id
                            where i.quelle = 'scraping' and i.importiert_am > now() - interval '1 day' order by f.name""")
            log("Ergebnis je Shop: Firma · Land · Status · Positionen · zugeordnet · Vorschläge · nicht zugeordnet · "
                "fehlerhaft · Dubletten")
            for r in cur.fetchall():
                log("  " + " · ".join(str(x) for x in r))
    log("FERTIG")


if __name__ == "__main__":
    run(sys.argv[1:] or None)

In [ ]:
%%writefile /content/priceclaw_bruecke.py
#!/usr/bin/env python3
"""
Grundfos-PIM – Import 11: Brücke PriceClaw (Supabase „solarics-produktdaten“, Tabelle public.offers) → marktbegleiter

OpenClaw liefert Scrapes an PriceClaw (offers je Quelle und Lieferung = file_id). Diese Brücke übernimmt die Lieferungen
der Marktbegleiter-Shops (QUELLEN) ins Schema marktbegleiter des Grundfos-PIM – über dieselbe Verarbeitung wie Import 9/10
(Prüfung, USt/EUR, Dubletten, Preis-Historie, Zuordnung zum Grundfos-Stamm).

  • nur herkunft = 'scrape' – eigene Einkaufspreise (eigener_ek) und Einzelfunde (live_fund) werden NIE übernommen
  • je Lieferung (file_id) genau ein Import; schon übernommene Lieferungen werden übersprungen → regelmäßig ausführbar
  • netto/brutto aus price_tax (gross/net), sonst Standard der Quelle
  • PriceClaw wird nur gelesen

Voraussetzung (Colab): Secrets SUPABASE_DB_URL (Grundfos-PIM) und SOLARICS_DB_URL (solarics-produktdaten, Session pooler).
"""

import json
import os
import re
import sys
from urllib.parse import urlparse

import psycopg

import marktbegleiter_import as mbi
import shop_exporte_import as sei
from datanorm_import import conn_params, db_url
from datenaudit import log

# (Name, Domain, Land, Standard netto/brutto, Suchmuster für Slug/Domain in quellen_stand) – Auswahl 27.10.2026
QUELLEN = [
    ("1A-Pumpen", "1a-pumpen.de", "DE", "brutto", "1a-pumpen"),
    ("ar-storeshop.com", "ar-storeshop.com", "IT", "brutto", "ar-storeshop"),
    ("BaumarktEU", "baumarkteu.de", "DE", "brutto", "baumarkteu"),
    ("ESC Grossiste", "esc-grossiste.fr", "FR", "brutto", "esc-grossiste"),
    ("glo24.de", "glo24.de", "DE", "brutto", "glo24"),
    ("heima24", "heima24.de", "DE", "brutto", "heima24"),
    ("heizung-billiger.de", "heizung-billiger.de", "DE", "brutto", "heizung-billiger"),
    ("heizungsdiscount24", "heizungsdiscount24.de", "DE", "brutto", "heizungsdiscount24"),
    ("Loebbeshop", "loebbeshop.de", "DE", "brutto", "loebbeshop"),
    ("Profluid", "profluid.de", "DE", "netto", "profluid"),
    ("Promoclima", "promoclima.it", "IT", "brutto", "promoclima"),
    ("PSW-Shop", "psw-shop.de", "DE", "netto", "psw-shop"),
    ("Pumpalia", "pumpalia.com", "ES", "brutto", "pumpalia"),
    ("pumpen24.de", "pumpen24.de", "DE", "brutto", "pumpen24"),
    ("pumpen-grosshandel.de", "pumpen-grosshandel.de", "DE", "brutto", "pumpen-grosshandel"),
    ("pumpen-guru.de", "pumpen-guru.de", "DE", "brutto", "pumpen-guru"),
    ("pumpen-heizung.de", "pumpen-heizung.de", "DE", "brutto", "pumpen-heizung"),
    ("Pumpendiscounter (Reiga GbR)", "pumpendiscounter.de", "DE", "brutto", "pumpendiscounter"),
    ("pumpengermany.de", "pumpengermany.de", "DE", "brutto", "pumpengermany"),
    ("Salvador Escoda", "salvadorescoda.com", "ES", "brutto", "salvador"),
    ("Sanundo", "sanundo.de", "DE", "brutto", "sanundo"),
    ("Sealing", "sealing.com.ua", "UA", "netto", "sealing"),
    ("Selfio", "selfio.de", "DE", "brutto", "selfio"),
    ("st-shop24.de", "st-shop24.de", "DE", "brutto", "st-shop24"),
    ("TEM Srl", "temsrl.it", "IT", "brutto", "temsrl"),
    ("TGA-Shop", "tga-shop.de", "DE", "netto", "tga-shop"),
    ("Weinmann & Schanz", "weinmann-schanz.de", "DE", "netto", "weinmann"),
    ("WIDA Handel", "wida-handel.de", "DE", "brutto", "wida"),
    ("shkhelden.de", "shkhelden.de", "DE", "brutto", "shkhelden"),
]
STEUER = {"gross": "brutto", "net": "netto"}


def quellen_finden(pc, muster):
    """source_ids in PriceClaw, deren Slug oder Domain zum Muster passt (Bindestrich/Unterstrich egal)."""
    rx = muster.replace("-", "[-_ ]?")
    with pc.cursor() as cur:
        cur.execute("""select source_id, slug, domain from public.quellen_stand
                        where slug ~* %s or coalesce(domain, '') ~* %s""", (rx, rx))
        return cur.fetchall()


def lieferungen(pc, source_ids):
    with pc.cursor() as cur:
        cur.execute("""select file_id, count(*), count(price_amount), max(scraped_at)::date, min(created_at)
                         from public.offers where source_id = any(%s) and herkunft = 'scrape' and file_id is not null
                        group by file_id order by min(created_at)""", (source_ids,))
        return cur.fetchall()


def zeile(o, land, standard_typ):
    (hersteller, mpn, ean, artikelnr, name, serie, preis, whg, steuer, verf, lieferzeit, kat, ukat, url, scraped) = o
    leer = lambda v: None if v is None or str(v).strip().lower() in sei.LEER else str(v).strip()  # noqa: E731
    f = {"hersteller": leer(hersteller), "mpn": leer(mpn), "ean": leer(ean), "bezeichnung": leer(name) or sei.name_aus_url(url),
         "baureihe": leer(serie), "kategorie": " > ".join(x for x in (leer(kat), leer(ukat)) if x) or None, "url": leer(url),
         "preis": float(preis) if preis is not None else None, "preis_typ": STEUER.get((steuer or "").lower(), standard_typ),
         "waehrung": (leer(whg) or "EUR")[:3].upper(), "lagerbestand": sei.lager(leer(verf)), "lieferzeit": leer(lieferzeit),
         "erfasst_am": scraped.date() if scraped else None, "shop_nr": leer(artikelnr)}
    ph = sei.pumpenhersteller(f["hersteller"], f["bezeichnung"])
    if not f["hersteller"] and ph:
        f["hersteller"] = ph
    if not f["mpn"] and ph == "Grundfos":           # Grundfos-Nummer aus Name, URL oder Shop-Nummer
        for q in (f["bezeichnung"], f["url"] and urlparse(f["url"]).path, f["shop_nr"]):
            m = sei.GF_NR.search(q or "")
            if m:
                f["mpn"] = m.group(1)
                break
    return f


def uebernehmen(pc, con, quelle, source_ids, file_id, n, n_preis, stand):
    name, domain, land, standard_typ, _ = quelle
    with con.cursor() as cur:
        cur.execute("select marktbegleiter.firma_ermitteln(%s, %s, %s)", (name, domain, land))
        firma_id = cur.fetchone()[0]
        kennung = f"priceclaw:{file_id}"
        cur.execute("""select i.id, (select count(*) from marktbegleiter.preise p where p.import_id = i.id)
                         from marktbegleiter.importe i where i.firma_id = %s and i.datei_sha256 = %s""", (firma_id, kennung))
        alt = cur.fetchone()
        if alt and alt[1] > 0:
            con.commit()
            return None
        # derselbe Scrape (gleicher Stand) schon über den Drive-Export (Import 10) übernommen? → nicht doppelt erfassen
        cur.execute("""select i.id, (select count(*) from marktbegleiter.preise p where p.import_id = i.id)
                         from marktbegleiter.importe i
                        where i.firma_id = %s and i.format = 'csv' and i.quelle = 'scraping' and i.stand = %s
                        order by 2 desc limit 1""", (firma_id, stand))
        drive = cur.fetchone()
        if drive and drive[1] >= 0.5 * n_preis:
            con.commit()
            return f"schon über Drive-Export übernommen (Import {drive[0]}, {drive[1]} Preise)"
        if alt:                                     # früherer Versuch ohne Preise → neu
            cur.execute("delete from marktbegleiter.import_zeilen where import_id = %s", (alt[0],))
            cur.execute("delete from marktbegleiter.importe where id = %s", (alt[0],))
        cur.execute("""insert into marktbegleiter.importe (firma_id, dateiname, datei_sha256, format, quelle, preisart, stand,
                                                          spalten, notiz)
                       values (%s, %s, %s, 'priceclaw', 'scraping', 'oeffentlich', %s, %s, %s) returning id""",
                    (firma_id, f"PriceClaw {domain} {stand}", kennung, stand,
                     json.dumps({"format": "priceclaw-offers", "file_id": str(file_id), "zeilen": n, "mit_preis": n_preis}),
                     "Import 11 (Brücke PriceClaw → marktbegleiter), nur herkunft = scrape"))
        import_id = cur.fetchone()[0]
        with pc.cursor(name=f"offers_{import_id}") as oc, \
                cur.copy("""COPY marktbegleiter.import_zeilen (import_id, zeile, rohdaten, hersteller, mpn, ean, bezeichnung,
                            baureihe, kategorie, url, preis, preis_typ, waehrung, lagerbestand, lieferzeit, erfasst_am,
                            quelle_url, land) FROM STDIN""") as cp:
            oc.itersize = 5000
            oc.execute("""select manufacturer_raw, mpn, ean, article_number, product_name, series, price_amount, currency,
                                 price_tax, availability_raw, delivery_time_raw, category_raw, subcategory_raw, product_url,
                                 scraped_at
                            from public.offers where source_id = any(%s) and file_id = %s and herkunft = 'scrape'
                           order by row_no nulls last, id""", (source_ids, file_id))
            for i, o in enumerate(oc, 1):
                f = zeile(o, land, standard_typ)
                roh = {k: f[k] for k in ("shop_nr", "hersteller", "mpn", "ean", "bezeichnung", "url") if f.get(k)}
                roh.update({"preis": f["preis"], "waehrung": f["waehrung"], "preis_typ": f["preis_typ"], "file_id": str(file_id)})
                cp.write_row((import_id, i, json.dumps(roh, ensure_ascii=False, default=str), f["hersteller"], f["mpn"],
                              f["ean"], f["bezeichnung"], f["baureihe"], f["kategorie"], f["url"], f["preis"], f["preis_typ"],
                              f["waehrung"], f["lagerbestand"], f["lieferzeit"], f["erfasst_am"] or stand, f["url"], land))
        cur.execute("select schritt, anzahl from marktbegleiter.import_verarbeiten(%s)", (import_id,))
        erg = dict(cur.fetchall())
        cur.execute("delete from marktbegleiter.import_zeilen where import_id = %s and status = 'ok'", (import_id,))
    con.commit()
    return erg


def run(nur=None):
    pc_url = os.environ.get("SOLARICS_DB_URL")
    if not pc_url:
        try:
            from google.colab import userdata  # type: ignore
            pc_url = userdata.get("SOLARICS_DB_URL")
        except Exception:  # noqa: BLE001
            pc_url = None
    if not pc_url:
        raise SystemExit("SOLARICS_DB_URL fehlt (Colab: Schlüssel-Symbol → Secret mit der Verbindungszeile von "
                         "„solarics-produktdaten“, Session pooler).")
    with psycopg.connect(**conn_params(pc_url.strip()), autocommit=False) as pc, \
            psycopg.connect(**conn_params(db_url()), autocommit=False) as con:
        pc.read_only = True                         # PriceClaw wird nur gelesen
        with con.cursor() as c:
            c.execute("set statement_timeout = 0")
        con.commit()
        for quelle in QUELLEN:
            if nur and quelle[1] not in nur:
                continue
            treffer = quellen_finden(pc, quelle[4])
            if not treffer:
                log(f"{quelle[0]}: in PriceClaw nicht vorhanden – übersprungen")
                continue
            ids = [t[0] for t in treffer]
            log(f"{quelle[0]} – PriceClaw-Quelle(n): {', '.join(t[1] for t in treffer)}")
            for file_id, n, n_preis, stand, _erstellt in lieferungen(pc, ids):
                if not n_preis:
                    log(f"  Lieferung {file_id}: {n} Zeilen ohne Preis – übersprungen")
                    continue
                try:
                    erg = uebernehmen(pc, con, quelle, ids, file_id, n, n_preis, stand)
                except Exception as e:  # noqa: BLE001 – eine kaputte Lieferung soll den Lauf nicht stoppen
                    con.rollback()
                    log(f"  ⚠ Fehler bei Lieferung {file_id}: {e!r}")
                    continue
                log(f"  Lieferung {file_id} ({stand}): " + ("schon übernommen" if erg is None else
                                                             erg if isinstance(erg, str) else f"{n} Zeilen · {erg}"))
            pc.rollback()                           # Lese-Transaktion je Quelle beenden
        with con.cursor() as cur:
            cur.execute("""select f.name, count(*) lieferungen, sum(i.positionen), sum(i.zugeordnet),
                                  sum((select count(*) from marktbegleiter.preise p where p.import_id = i.id))
                             from marktbegleiter.importe i join marktbegleiter.firmen f on f.id = i.firma_id
                            where i.format = 'priceclaw' group by f.name order by f.name""")
            log("Übernommen aus PriceClaw: Firma · Lieferungen · Positionen · zugeordnet · Preise")
            for r in cur.fetchall():
                log("  " + " · ".join(str(x) for x in r))
    log("FERTIG")


if __name__ == "__main__":
    run(sys.argv[1:] or None)

In [ ]:
import os, sys, importlib
os.environ['SUPABASE_DB_URL'] = userdata.get('SUPABASE_DB_URL')
os.environ['SOLARICS_DB_URL'] = userdata.get('SOLARICS_DB_URL')
sys.path.insert(0, '/content')
import datenaudit, datanorm_import, projektablage_import, marktbegleiter_import, shop_exporte_import, priceclaw_bruecke
for m in (datenaudit, datanorm_import, projektablage_import, marktbegleiter_import, shop_exporte_import, priceclaw_bruecke):
    importlib.reload(m)
priceclaw_bruecke.run()

## Fertig – die Übersicht am Ende an Claude schicken.